In [ ]:
import os
# Configure before running. See README; use a fresh output tree.
os.environ.setdefault("EGLIMUC_WORKSPACE", "/content/eglimuc")
os.environ.setdefault("EGLIMUC_MOUNT_DRIVE", "0")


# Notebook 13 v2: corrected copy

One execution flow, based on the original training cell, with the later recovery cell's ID-safe label join and saved-prediction validation integrated before dispatch. The original file is untouched.

**Default: `EXECUTION_MODE = "ASSEMBLE"`.** Configure the first setup cell, then run the workflow code cell to validate the frozen plan and all five completed folds, then rebuild final results from the 30 saved blind prediction files. No training is dispatched in this mode. It still imports the original ML dependencies and may install missing dependencies, optionally mounts Drive only when `EGLIMUC_MOUNT_DRIVE=1`, and writes the original manifests, result CSVs and completion receipt to the configured paths, plus a prediction-validation manifest. It does not reuse existing final CSVs as a shortcut. Back up existing outputs if their exact bytes or timestamps must be retained. Fresh runtime recommended.

To intentionally train/resume, change the mode to `"RUN"`; `"PLAN"` retains the original plan-validation behavior. Both the original scientific specification and training/selection/aggregation/bootstrap calculations are retained. The frozen analysis-plan fingerprint is `5c5cab052235f11e10a43f0fc23cdc8831a89b502bcbefa85973ad6e5d076cd6`.

Corrections: preserve prediction `patient_id`/`cv_fold` during the frozen-label join and verify them against the vault; reject malformed/incomplete prediction inputs; validate outer completion and selection receipts; reject missing or mismatched outer checkpoint fingerprints before restoring state. Duplicate recovery cells and stale outputs have been removed.

Initial validation of the corrected source before release cleanup: Python syntax, notebook structure, synthetic label-join and receipt/checkpoint rejection tests, and mocked dispatcher tests. No real data, GPU training, end-to-end run, Drive write, or historical output-hash comparison was performed while preparing the copy. A later audit verified the actual 30 saved-prediction label joins, endpoint rows, and bootstrap results; see the repository README for its scope. Release cleanup received static and synthetic checks only. Neither stage claims full training/inference reproducibility.

Original and sanitized source hashes are recorded in `docs/SOURCE_PROVENANCE.json`.


In [ ]:
# ==================================================================================================
# 13_nested_patient_grouped_evaluation_v2.ipynb
# CORRECTED SINGLE-FLOW COLAB NOTEBOOK (ASSEMBLE by default)
# AUTOMATIC OUTER-FOLD EXECUTION: 0 -> 4
# ==================================================================================================
#
# PURPOSE
# -------
# Genuine nested patient-grouped development evaluation.
#
# OUTER EVALUATION
# ----------------
# Five frozen patient folds.
#
# INNER SELECTION WITHIN EACH OUTER-TRAINING PARTITION
# ----------------------------------------------------
# - Four patient-grouped inner validation folds
# - Five ViT loss candidates screened using seed 42
# - Top two ViT candidates confirmed using seeds 42, 2025, 2026
# - One of six patient-aggregation rules selected using only inner OOF predictions
# - Five Swin loss candidates screened using seed 42
# - Top two Swin candidates confirmed using seeds 42, 2025, 2026
# - One ViT and one Swin configuration selected
# - ViT-versus-Swin system selection performed using inner results only
# - Outer-refit epoch count derived from inner best epochs only
#
# OUTER REFIT AND EVALUATION
# --------------------------
# - Selected ViT and Swin configurations are refitted on the complete outer-training partition
# - No outer-fold validation or checkpoint selection occurs
# - Outer inference files are saved WITHOUT y_true or patient_y_true
# - Labels are joined only after all five outer folds are complete
# - No partial outer-fold performance results are generated
#
# ANALYTICAL OUTPUTS
# ------------------
# 1. Nested ViT-family estimate
# 2. Nested Swin-family estimate
# 3. Fully nested selected-procedure estimate
#
# V2 GUARANTEES
# -------------
# - No official-partition path is defined
# - No official-partition data are loaded
# - No outer patient contributes to inner training or validation
# - No outer label contributes to checkpoint selection
# - No outer label contributes to loss selection
# - No outer label contributes to aggregation selection
# - No outer label contributes to system selection
# - No outer label contributes to outer-refit epoch selection
# - Fold-level outer predictions contain no explicit true-label columns
# - Publication metrics are generated only after all five outer folds are complete
#
# COMPUTATIONAL WORKLOAD
# ----------------------
# Maximum inner fits:
#     5 outer folds x 72 inner fits = 360
#
# Maximum outer refits:
#     5 outer folds x 2 systems x 3 seeds = 30
#
# Maximum total:
#     390 model fits
#
# RESUMPTION
# ----------
# This notebook is interruption-safe:
# - completed inner runs are reused
# - incomplete inner runs resume from last_epoch_checkpoint.pt
# - completed outer refits are reused
# - completed outer folds are skipped
#
# To train/resume, explicitly set EXECUTION_MODE = "RUN".
# Default ASSEMBLE only rebuilds results from completed saved predictions.
# ASSEMBLE can write manifests/results/receipts to the configured Drive paths.
#
# ==================================================================================================


# ==================================================================================================
# 0. Lightweight dependency installation
# ==================================================================================================

import sys
import subprocess
import importlib.util


REQUIRED_PACKAGES = {
    "timm": "timm==1.0.28",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "PIL": "Pillow",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
}


missing_packages = [
    package_name
    for import_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(import_name) is None
]


if missing_packages:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            *missing_packages,
        ]
    )


# ==================================================================================================
# 1. Imports
# ==================================================================================================

import gc
import re
import io
import os
import json
import math
import time
import copy
import random
import hashlib
import platform
import warnings

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd

from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torchvision

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm

from sklearn.metrics import (
    cohen_kappa_score,
    mean_absolute_error,
    f1_score,
    accuracy_score,
    confusion_matrix,
)

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display


warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True


# ==================================================================================================
# 2. OPERATIONAL CONTROLS
# ==================================================================================================
#
# The scientific plan is unchanged.
#
# RUN:
#     automatically process folds 0 -> 4
#
# PLAN:
#     plan validation only
#
# ASSEMBLE:
#     no training; requires all folds complete
#
# ==================================================================================================

EXECUTION_MODE = "ASSEMBLE"

# Automatically processes 0, 1, 2, 3, 4.
# Completed folds will be skipped.
OUTER_FOLDS_TO_RUN = list(range(5))

VERIFY_ALL_IMAGE_PATHS_IN_PLAN = True
VERIFY_PRETRAINED_WEIGHTS_IN_PLAN = True

USE_LOCAL_IMAGE_CACHE = True

KEEP_COMPLETED_INNER_CHECKPOINTS = False
KEEP_COMPLETED_OUTER_CHECKPOINTS = False

SHOW_FINAL_FIGURES = True


ALLOWED_EXECUTION_MODES = {
    "PLAN",
    "RUN",
    "ASSEMBLE",
}


EXECUTION_MODE = str(
    EXECUTION_MODE
).strip().upper()


if EXECUTION_MODE not in ALLOWED_EXECUTION_MODES:
    raise ValueError(
        f"EXECUTION_MODE must be one of "
        f"{sorted(ALLOWED_EXECUTION_MODES)}."
    )


# ==================================================================================================
# 3. Development-only runtime guardrail
# ==================================================================================================

FORBIDDEN_RUNTIME_OBJECTS = [
    "TEST_ROOT",
    "TEST_PATH",
    "TEST_DIR",
    "OFFICIAL_TEST_ROOT",
    "OFFICIAL_PARTITION_ROOT",
    "test_df",
    "test_dataset",
    "test_loader",
    "official_df",
    "official_loader",
    "ensemble_test_df",
    "test_predictions",
]


present_forbidden_objects = [
    object_name
    for object_name in FORBIDDEN_RUNTIME_OBJECTS
    if object_name in globals()
]


if present_forbidden_objects:
    raise RuntimeError(
        "Official-partition objects were already present in this runtime:\n"
        f"{present_forbidden_objects}\n\n"
        "Restart the runtime and execute only this notebook."
    )


print("=" * 125)
print("13_nested_patient_grouped_evaluation_v2")
print("=" * 125)
print("Execution mode:", EXECUTION_MODE)
print("Requested outer folds:", OUTER_FOLDS_TO_RUN)
print("Official-partition path defined: False")
print("Official-partition files loaded: False")
print("Official-partition labels accessed: False")


# ==================================================================================================
# 4. Mount Google Drive
# ==================================================================================================

try:
    from google.colab import drive

    (drive.mount(
        "/content/drive",
        force_remount=False,
    ) if os.environ.get("EGLIMUC_MOUNT_DRIVE", "0") == "1" else None)

except Exception as mount_error:
    print("Google Drive was not mounted automatically.")
    print("Reason:", repr(mount_error))


# ==================================================================================================
# 5. Paths
# ==================================================================================================

DRIVE_ROOT_CANDIDATES = [
    Path((os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '')),
    Path((os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '')),
]


DRIVE_ROOT = next(
    (
        path
        for path in DRIVE_ROOT_CANDIDATES
        if path.exists()
    ),
    None,
)


if DRIVE_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the Google Drive root.\n"
        + "\n".join(
            str(path)
            for path in DRIVE_ROOT_CANDIDATES
        )
    )


PROJECT_ROOT = DRIVE_ROOT / "LIMUC"

DEVELOPMENT_ROOT = (
    PROJECT_ROOT
    / "train_and_validation_sets"
)

REVISION_ROOT = (
    PROJECT_ROOT
    / "ViT2"
    / "revision_2026"
)


NOTEBOOK01_ROOT = (
    REVISION_ROOT
    / "01_vit_loss_screening_development_only"
)


NOTEBOOK03_ROOT = (
    REVISION_ROOT
    / "03_patient_aggregation_and_reference_sensitivity"
)


NOTEBOOK06_CANDIDATES = [
    REVISION_ROOT
    / "06_backbone_benchmark_development_only_v1",

    REVISION_ROOT
    / "06_backbone_benchmark_development_only",
]


NOTEBOOK06_ROOT = next(
    (
        path
        for path in NOTEBOOK06_CANDIDATES
        if path.exists()
    ),
    None,
)


if NOTEBOOK06_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the Notebook 06 output directory.\n"
        + "\n".join(
            str(path)
            for path in NOTEBOOK06_CANDIDATES
        )
    )


DEVELOPMENT_MANIFEST_PATH = (
    NOTEBOOK01_ROOT
    / "development_image_manifest_with_folds.csv"
)


PATIENT_FOLD_ASSIGNMENT_PATH = (
    NOTEBOOK01_ROOT
    / "patient_fold_assignment.csv"
)


PATIENT_REFERENCE_AUDIT_CANDIDATES = [
    NOTEBOOK03_ROOT
    / "patient_reference_rule_sensitivity.csv",

    NOTEBOOK01_ROOT
    / "patient_reference_rule_sensitivity.csv",
]


PATIENT_REFERENCE_AUDIT_PATH = next(
    (
        path
        for path in PATIENT_REFERENCE_AUDIT_CANDIDATES
        if path.exists()
    ),
    None,
)


if PATIENT_REFERENCE_AUDIT_PATH is None:
    raise FileNotFoundError(
        "Could not locate patient_reference_rule_sensitivity.csv.\n"
        + "\n".join(
            str(path)
            for path in PATIENT_REFERENCE_AUDIT_CANDIDATES
        )
    )


NOTEBOOK06_PLAN_PATH = (
    NOTEBOOK06_ROOT
    / "analysis_plan_frozen_before_backbone_benchmark.json"
)


V1_ROOT = (
    REVISION_ROOT
    / "13_nested_patient_grouped_evaluation_v1"
)


V2_ROOT = (
    REVISION_ROOT
    / "13_nested_patient_grouped_evaluation_v2"
)


INNER_ROOT = (
    V2_ROOT
    / "inner_selection"
)


OUTER_ROOT = (
    V2_ROOT
    / "outer_refits_blind"
)


FINAL_ROOT = (
    V2_ROOT
    / "final_nested_results"
)


MANIFEST_ROOT = (
    V2_ROOT
    / "manifests"
)


FIGURE_ROOT = (
    V2_ROOT
    / "figures"
)


FIGURE_PNG_ROOT = (
    FIGURE_ROOT
    / "png"
)


FIGURE_SVG_ROOT = (
    FIGURE_ROOT
    / "svg"
)


FIGURE_PDF_ROOT = (
    FIGURE_ROOT
    / "pdf"
)


SOURCE_DATA_ROOT = (
    V2_ROOT
    / "source_data"
)


for directory in [
    V2_ROOT,
    INNER_ROOT,
    OUTER_ROOT,
    FINAL_ROOT,
    MANIFEST_ROOT,
    FIGURE_ROOT,
    FIGURE_PNG_ROOT,
    FIGURE_SVG_ROOT,
    FIGURE_PDF_ROOT,
    SOURCE_DATA_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


LOCAL_IMAGE_CACHE_ROOT = Path(
    "/content/limuc_nested_v2_image_cache"
)


LOCAL_IMAGE_CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


REQUIRED_INPUT_PATHS = [
    DEVELOPMENT_ROOT,
    DEVELOPMENT_MANIFEST_PATH,
    PATIENT_FOLD_ASSIGNMENT_PATH,
    PATIENT_REFERENCE_AUDIT_PATH,
    NOTEBOOK06_PLAN_PATH,
]


for required_path in REQUIRED_INPUT_PATHS:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Missing required development-only input:\n{required_path}"
        )


print("\nDevelopment root:", DEVELOPMENT_ROOT)
print("Development manifest:", DEVELOPMENT_MANIFEST_PATH)
print("Patient-fold assignment:", PATIENT_FOLD_ASSIGNMENT_PATH)
print("Patient-reference audit:", PATIENT_REFERENCE_AUDIT_PATH)
print("Notebook 06 plan:", NOTEBOOK06_PLAN_PATH)
print("Version 2 output root:", V2_ROOT)


# ==================================================================================================
# 6. Frozen scientific settings
# ==================================================================================================

PROTOCOL_VERSION = (
    "nested_patient_grouped_evaluation_v2"
)

NUM_CLASSES = 4

CLASS_IDS = [
    0,
    1,
    2,
    3,
]

NUM_OUTER_FOLDS = 5
NUM_INNER_FOLDS_PER_OUTER = 4

EXPECTED_DEVELOPMENT_IMAGES = 9590
EXPECTED_DEVELOPMENT_PATIENTS = 478

IMAGE_SIZE = 384

TRAIN_BATCH_SIZE = 4

GRADIENT_ACCUMULATION_STEPS = 2

EFFECTIVE_TRAIN_BATCH_SIZE = (
    TRAIN_BATCH_SIZE
    *
    GRADIENT_ACCUMULATION_STEPS
)

VALIDATION_BATCH_SIZE = 8

NUM_WORKERS = 0

MAX_EPOCHS = 25

EARLY_STOP_PATIENCE = 8

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

WARMUP_EPOCHS = 3

MIN_LR_RATIO = 0.05

LABEL_SMOOTHING = 0.02

DROP_RATE = 0.10

DROP_PATH_RATE = 0.05

GRADIENT_CLIP_NORM = 1.0

USE_MIXED_PRECISION = True

USE_VALIDATION_TTA = True

TTA_HORIZONTAL_FLIP = True

USE_IMAGE_CLASS_WEIGHTS = True

SCREENING_SEED = 42

CONFIRMATION_SEEDS = [
    42,
    2025,
    2026,
]

PRIMARY_REFERENCE_RULE = (
    "modal_low"
)

CHECKPOINT_AGGREGATION_RULE = (
    "mean_probability_argmax"
)

BOOTSTRAP_REPLICATES = 5000

BOOTSTRAP_SEED = 20260905

IMAGE_IO_MAX_RETRIES = 5

IMAGE_IO_INITIAL_DELAY_SECONDS = 1.0

IMAGE_IO_MAX_DELAY_SECONDS = 20.0

DRIVE_IO_MAX_RETRIES = 5

DRIVE_IO_INITIAL_DELAY_SECONDS = 1.0


VIT_MODEL_NAME = (
    "vit_base_patch16_384.augreg_in21k_ft_in1k"
)


SWIN_MODEL_NAME = (
    "swin_base_patch4_window12_384.ms_in22k_ft_in1k"
)


MODEL_FAMILIES = {
    "ViT": {
        "model_name": (
            VIT_MODEL_NAME
        ),
    },
    "Swin": {
        "model_name": (
            SWIN_MODEL_NAME
        ),
    },
}


LOSS_TEMPLATES = [
    {
        "loss_code": "L1",
        "loss_name": "cross_entropy_only",
        "ordinal_kind": "none",
        "ordinal_alpha": 1.0,
        "ordinal_weight": 0.0,
    },
    {
        "loss_code": "L2",
        "loss_name": "expected_value_ordinal_mae",
        "ordinal_kind": "expected_value_mae",
        "ordinal_alpha": 1.0,
        "ordinal_weight": 0.10,
    },
    {
        "loss_code": "L3",
        "loss_name": "expected_ordinal_distance",
        "ordinal_kind": "expected_distance",
        "ordinal_alpha": 1.0,
        "ordinal_weight": 0.10,
    },
    {
        "loss_code": "L4",
        "loss_name": "expected_ordinal_distance",
        "ordinal_kind": "expected_distance",
        "ordinal_alpha": 2.0,
        "ordinal_weight": 0.10,
    },
    {
        "loss_code": "L5",
        "loss_name": "expected_ordinal_distance",
        "ordinal_kind": "expected_distance",
        "ordinal_alpha": 1.0,
        "ordinal_weight": 0.20,
    },
]


CANDIDATES = []


for family_name, family_specification in MODEL_FAMILIES.items():

    for loss_template in LOSS_TEMPLATES:

        candidate_id = (
            family_name.lower()
            +
            "_"
            +
            loss_template[
                "loss_code"
            ].lower()
            +
            "_"
            +
            loss_template[
                "loss_name"
            ]
            +
            "_a"
            +
            str(
                int(
                    loss_template[
                        "ordinal_alpha"
                    ]
                )
            )
            +
            "_w"
            +
            f"{int(round(loss_template['ordinal_weight'] * 100)):03d}"
        )

        CANDIDATES.append(
            {
                "candidate_id": (
                    candidate_id
                ),
                "family": (
                    family_name
                ),
                "model_name": (
                    family_specification[
                        "model_name"
                    ]
                ),
                **loss_template,
            }
        )


CANDIDATE_LOOKUP = {
    candidate[
        "candidate_id"
    ]: candidate
    for candidate in CANDIDATES
}


VIT_CANDIDATE_IDS = [
    candidate[
        "candidate_id"
    ]
    for candidate in CANDIDATES
    if candidate[
        "family"
    ]
    ==
    "ViT"
]


SWIN_CANDIDATE_IDS = [
    candidate[
        "candidate_id"
    ]
    for candidate in CANDIDATES
    if candidate[
        "family"
    ]
    ==
    "Swin"
]


CANDIDATE_ORDER = {
    candidate_id: index
    for index, candidate_id in enumerate(
        [
            *VIT_CANDIDATE_IDS,
            *SWIN_CANDIDATE_IDS,
        ]
    )
}


AGGREGATION_RULES = [
    "mean_probability_argmax",
    "median_probability_argmax",
    "majority_vote_low_tie",
    "mean_expected_mes_round",
    "median_expected_mes_round",
    "maximum_predicted_image_class",
]


AGGREGATION_LABELS = {
    "mean_probability_argmax": (
        "Mean class probability + argmax"
    ),
    "median_probability_argmax": (
        "Median class probability + argmax"
    ),
    "majority_vote_low_tie": (
        "Majority vote, lower-class tie"
    ),
    "mean_expected_mes_round": (
        "Mean expected MES, half-up rounding"
    ),
    "median_expected_mes_round": (
        "Median expected MES, half-up rounding"
    ),
    "maximum_predicted_image_class": (
        "Maximum predicted image class"
    ),
}


AGGREGATION_ORDER = {
    aggregation_rule: index
    for index, aggregation_rule in enumerate(
        AGGREGATION_RULES
    )
}


PROBABILITY_COLUMNS = [
    "prob_0",
    "prob_1",
    "prob_2",
    "prob_3",
]


METRIC_NAMES = [
    "qwk",
    "mae",
    "macro_f1",
    "accuracy",
    "balanced_accuracy",
]


# ==================================================================================================
# 7. Generic helpers
# ==================================================================================================

def utc_now():

    return datetime.now(
        timezone.utc
    ).isoformat()


def safe_package_version(
    package_name,
):

    try:
        return version(
            package_name
        )

    except PackageNotFoundError:
        return None


def canonical_json_bytes(
    payload,
):

    return json.dumps(
        payload,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
        ensure_ascii=False,
        default=str,
    ).encode(
        "utf-8"
    )


def dictionary_sha256(
    payload,
):

    return hashlib.sha256(
        canonical_json_bytes(
            payload
        )
    ).hexdigest()


def file_sha256(
    path,
    block_size=8 * 1024 * 1024,
):

    path = Path(
        path
    )

    hasher = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:

        while True:

            block = file.read(
                block_size
            )

            if not block:
                break

            hasher.update(
                block
            )

    return hasher.hexdigest()


def values_sha256(
    values,
):

    normalized_values = [
        str(
            value
        )
        for value in values
    ]

    return dictionary_sha256(
        normalized_values
    )


def dataframe_key_sha256(
    dataframe,
    columns,
):

    normalized_dataframe = (
        dataframe[
            columns
        ]
        .copy()
        .astype(
            str
        )
        .sort_values(
            columns
        )
        .reset_index(
            drop=True
        )
    )

    return hashlib.sha256(
        normalized_dataframe.to_csv(
            index=False
        ).encode(
            "utf-8"
        )
    ).hexdigest()


def retry_delay(
    attempt_index,
    initial_delay,
    maximum_delay=20.0,
):

    return min(
        maximum_delay,
        initial_delay
        *
        (
            2
            **
            attempt_index
        ),
    )


def temporary_path(
    path,
):

    path = Path(
        path
    )

    return path.with_name(
        path.name
        +
        f".{os.getpid()}.{time.time_ns()}.temporary"
    )


def atomic_csv_save(
    dataframe,
    path,
    index=False,
):

    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    last_error = None

    for attempt_index in range(
        DRIVE_IO_MAX_RETRIES
    ):

        temp_path = temporary_path(
            path
        )

        try:

            dataframe.to_csv(
                temp_path,
                index=index,
            )

            os.replace(
                temp_path,
                path,
            )

            return

        except OSError as error:

            last_error = error

            try:

                if temp_path.exists():
                    temp_path.unlink()

            except Exception:
                pass

            time.sleep(
                retry_delay(
                    attempt_index,
                    DRIVE_IO_INITIAL_DELAY_SECONDS,
                )
            )

    raise OSError(
        f"Could not save CSV:\n{path}\n{repr(last_error)}"
    )


def atomic_json_save(
    payload,
    path,
):

    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    last_error = None

    for attempt_index in range(
        DRIVE_IO_MAX_RETRIES
    ):

        temp_path = temporary_path(
            path
        )

        try:

            with open(
                temp_path,
                "w",
                encoding="utf-8",
            ) as file:

                json.dump(
                    payload,
                    file,
                    indent=2,
                    ensure_ascii=False,
                    default=str,
                )

                file.flush()

                os.fsync(
                    file.fileno()
                )

            os.replace(
                temp_path,
                path,
            )

            return

        except OSError as error:

            last_error = error

            try:

                if temp_path.exists():
                    temp_path.unlink()

            except Exception:
                pass

            time.sleep(
                retry_delay(
                    attempt_index,
                    DRIVE_IO_INITIAL_DELAY_SECONDS,
                )
            )

    raise OSError(
        f"Could not save JSON:\n{path}\n{repr(last_error)}"
    )


def atomic_text_save(
    text,
    path,
):

    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    last_error = None

    for attempt_index in range(
        DRIVE_IO_MAX_RETRIES
    ):

        temp_path = temporary_path(
            path
        )

        try:

            with open(
                temp_path,
                "w",
                encoding="utf-8",
            ) as file:

                file.write(
                    text
                )

                file.flush()

                os.fsync(
                    file.fileno()
                )

            os.replace(
                temp_path,
                path,
            )

            return

        except OSError as error:

            last_error = error

            try:

                if temp_path.exists():
                    temp_path.unlink()

            except Exception:
                pass

            time.sleep(
                retry_delay(
                    attempt_index,
                    DRIVE_IO_INITIAL_DELAY_SECONDS,
                )
            )

    raise OSError(
        f"Could not save text:\n{path}\n{repr(last_error)}"
    )


def atomic_torch_save(
    payload,
    path,
):

    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    last_error = None

    for attempt_index in range(
        DRIVE_IO_MAX_RETRIES
    ):

        temp_path = temporary_path(
            path
        )

        try:

            torch.save(
                payload,
                temp_path,
            )

            os.replace(
                temp_path,
                path,
            )

            return

        except OSError as error:

            last_error = error

            try:

                if temp_path.exists():
                    temp_path.unlink()

            except Exception:
                pass

            time.sleep(
                retry_delay(
                    attempt_index,
                    DRIVE_IO_INITIAL_DELAY_SECONDS,
                )
            )

    raise OSError(
        f"Could not save checkpoint:\n{path}\n{repr(last_error)}"
    )


def safe_torch_load(
    path,
    map_location="cpu",
):

    path = Path(
        path
    )

    last_error = None

    for attempt_index in range(
        DRIVE_IO_MAX_RETRIES
    ):

        try:

            try:

                return torch.load(
                    path,
                    map_location=(
                        map_location
                    ),
                    weights_only=False,
                )

            except TypeError:

                return torch.load(
                    path,
                    map_location=(
                        map_location
                    ),
                )

        except OSError as error:

            last_error = error

            time.sleep(
                retry_delay(
                    attempt_index,
                    DRIVE_IO_INITIAL_DELAY_SECONDS,
                )
            )

    raise OSError(
        f"Could not load checkpoint:\n{path}\n{repr(last_error)}"
    )


for abandoned_file in V2_ROOT.rglob(
    "*.temporary"
):

    try:
        abandoned_file.unlink()

    except Exception:
        pass


# ==================================================================================================
# 8. Load and validate development data
# ==================================================================================================

development_df = pd.read_csv(
    DEVELOPMENT_MANIFEST_PATH
)


REQUIRED_DEVELOPMENT_COLUMNS = {
    "relative_path",
    "label",
    "patient_id",
    "cv_fold",
    "patient_y_true",
}


missing_columns = (
    REQUIRED_DEVELOPMENT_COLUMNS
    -
    set(
        development_df.columns
    )
)


if missing_columns:

    raise RuntimeError(
        "Development manifest is missing columns:\n"
        f"{sorted(missing_columns)}"
    )


development_df[
    "relative_path"
] = development_df[
    "relative_path"
].astype(
    str
)


development_df[
    "path"
] = development_df[
    "relative_path"
].map(
    lambda relative_path: str(
        DEVELOPMENT_ROOT
        /
        Path(
            relative_path
        )
    )
)


for integer_column in [
    "label",
    "patient_id",
    "cv_fold",
    "patient_y_true",
]:

    development_df[
        integer_column
    ] = (
        pd.to_numeric(
            development_df[
                integer_column
            ],
            errors="raise",
        )
        .astype(
            int
        )
    )


development_df = (
    development_df.sort_values(
        [
            "patient_id",
            "relative_path",
        ]
    )
    .reset_index(
        drop=True
    )
)


if len(
    development_df
) != EXPECTED_DEVELOPMENT_IMAGES:

    raise RuntimeError(
        f"Expected {EXPECTED_DEVELOPMENT_IMAGES} development images; "
        f"found {len(development_df)}."
    )


if (
    development_df[
        "patient_id"
    ].nunique()
    !=
    EXPECTED_DEVELOPMENT_PATIENTS
):

    raise RuntimeError(
        f"Expected {EXPECTED_DEVELOPMENT_PATIENTS} development patients."
    )


if development_df[
    "relative_path"
].duplicated().any():

    duplicated_paths = (
        development_df.loc[
            development_df[
                "relative_path"
            ].duplicated(
                keep=False
            ),
            "relative_path",
        ]
        .tolist()
    )

    raise RuntimeError(
        "Duplicate relative paths were found:\n"
        +
        "\n".join(
            duplicated_paths[
                :20
            ]
        )
    )


if set(
    development_df[
        "label"
    ].unique()
) != set(
    CLASS_IDS
):

    raise RuntimeError(
        "Image labels are not exactly 0, 1, 2, 3."
    )


if set(
    development_df[
        "patient_y_true"
    ].unique()
) != set(
    CLASS_IDS
):

    raise RuntimeError(
        "Patient references are not exactly 0, 1, 2, 3."
    )


if set(
    development_df[
        "cv_fold"
    ].unique()
) != set(
    range(
        NUM_OUTER_FOLDS
    )
):

    raise RuntimeError(
        "Frozen fold identifiers are not exactly 0–4."
    )


if (
    development_df.groupby(
        "patient_id"
    )[
        "patient_y_true"
    ]
    .nunique()
    .max()
    !=
    1
):

    raise RuntimeError(
        "A patient has inconsistent patient_y_true values."
    )


if (
    development_df.groupby(
        "patient_id"
    )[
        "cv_fold"
    ]
    .nunique()
    .max()
    !=
    1
):

    raise RuntimeError(
        "A patient occurs in multiple frozen folds."
    )


patient_manifest_df = (
    development_df[
        [
            "patient_id",
            "patient_y_true",
            "cv_fold",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "patient_id"
    )
    .reset_index(
        drop=True
    )
)


if len(
    patient_manifest_df
) != EXPECTED_DEVELOPMENT_PATIENTS:

    raise RuntimeError(
        "Patient manifest does not contain one row per patient."
    )


# ==================================================================================================
# 9. Validate patient-fold assignment
# ==================================================================================================

fold_assignment_df = pd.read_csv(
    PATIENT_FOLD_ASSIGNMENT_PATH
)


required_fold_columns = {
    "patient_id",
    "cv_fold",
}


if not required_fold_columns.issubset(
    fold_assignment_df.columns
):

    raise RuntimeError(
        "Patient-fold assignment file must contain patient_id and cv_fold."
    )


fold_assignment_df[
    "patient_id"
] = (
    pd.to_numeric(
        fold_assignment_df[
            "patient_id"
        ],
        errors="raise",
    )
    .astype(
        int
    )
)


fold_assignment_df[
    "cv_fold"
] = (
    pd.to_numeric(
        fold_assignment_df[
            "cv_fold"
        ],
        errors="raise",
    )
    .astype(
        int
    )
)


fold_validation = (
    patient_manifest_df.merge(
        fold_assignment_df[
            [
                "patient_id",
                "cv_fold",
            ]
        ],
        on="patient_id",
        how="inner",
        suffixes=(
            "_manifest",
            "_assignment",
        ),
        validate="one_to_one",
    )
)


if len(
    fold_validation
) != EXPECTED_DEVELOPMENT_PATIENTS:

    raise RuntimeError(
        "Patient-fold assignment alignment failed."
    )


if not (
    fold_validation[
        "cv_fold_manifest"
    ]
    ==
    fold_validation[
        "cv_fold_assignment"
    ]
).all():

    raise RuntimeError(
        "Frozen fold assignments differ between source files."
    )


# ==================================================================================================
# 10. Validate modal-low reference
# ==================================================================================================

reference_audit_df = pd.read_csv(
    PATIENT_REFERENCE_AUDIT_PATH
)


if not {
    "patient_id",
    PRIMARY_REFERENCE_RULE,
}.issubset(
    reference_audit_df.columns
):

    raise RuntimeError(
        "Patient-reference audit must contain patient_id and modal_low."
    )


reference_audit_df[
    "patient_id"
] = (
    pd.to_numeric(
        reference_audit_df[
            "patient_id"
        ],
        errors="raise",
    )
    .astype(
        int
    )
)


reference_audit_df[
    PRIMARY_REFERENCE_RULE
] = (
    pd.to_numeric(
        reference_audit_df[
            PRIMARY_REFERENCE_RULE
        ],
        errors="raise",
    )
    .astype(
        int
    )
)


reference_validation = (
    patient_manifest_df.merge(
        reference_audit_df[
            [
                "patient_id",
                PRIMARY_REFERENCE_RULE,
            ]
        ],
        on="patient_id",
        how="inner",
        validate="one_to_one",
    )
)


if len(
    reference_validation
) != EXPECTED_DEVELOPMENT_PATIENTS:

    raise RuntimeError(
        "Patient-reference audit alignment failed."
    )


if not (
    reference_validation[
        "patient_y_true"
    ]
    ==
    reference_validation[
        PRIMARY_REFERENCE_RULE
    ]
).all():

    raise RuntimeError(
        "Manifest patient_y_true differs from frozen modal-low."
    )


# ==================================================================================================
# 11. Frozen outer-label vault
# ==================================================================================================

outer_label_vault_df = (
    development_df[
        [
            "relative_path",
            "patient_id",
            "cv_fold",
            "label",
            "patient_y_true",
        ]
    ]
    .copy()
    .sort_values(
        "relative_path"
    )
    .reset_index(
        drop=True
    )
)


OUTER_LABEL_VAULT_FINGERPRINT = (
    dataframe_key_sha256(
        outer_label_vault_df,
        [
            "relative_path",
            "patient_id",
            "cv_fold",
            "label",
            "patient_y_true",
        ],
    )
)


DEVELOPMENT_MANIFEST_FINGERPRINT = (
    dataframe_key_sha256(
        development_df,
        [
            "relative_path",
            "label",
            "patient_id",
            "cv_fold",
            "patient_y_true",
        ],
    )
)


fold_patient_counts = (
    patient_manifest_df[
        "cv_fold"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
)


fold_class_counts = (
    patient_manifest_df.groupby(
        [
            "cv_fold",
            "patient_y_true",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        index=range(
            NUM_OUTER_FOLDS
        ),
        columns=CLASS_IDS,
        fill_value=0,
    )
)


print(
    "\nDevelopment images:",
    len(
        development_df
    ),
)


print(
    "Development patients:",
    development_df[
        "patient_id"
    ].nunique(),
)


print(
    "Outer-fold patient counts:",
    fold_patient_counts,
)


print(
    "\nOuter-fold class counts:"
)


display(
    fold_class_counts
)


# ==================================================================================================
# 12. Partition separation audit
# ==================================================================================================

partition_audit_rows = []


for outer_fold in range(
    NUM_OUTER_FOLDS
):

    outer_patients = set(
        patient_manifest_df.loc[
            patient_manifest_df[
                "cv_fold"
            ]
            ==
            outer_fold,
            "patient_id",
        ]
        .astype(
            int
        )
        .tolist()
    )


    inner_fold_ids = [
        fold_id
        for fold_id in range(
            NUM_OUTER_FOLDS
        )
        if fold_id
        !=
        outer_fold
    ]


    for inner_validation_fold in inner_fold_ids:

        inner_validation_patients = set(
            patient_manifest_df.loc[
                patient_manifest_df[
                    "cv_fold"
                ]
                ==
                inner_validation_fold,
                "patient_id",
            ]
            .astype(
                int
            )
            .tolist()
        )


        inner_training_patients = set(
            patient_manifest_df.loc[
                ~patient_manifest_df[
                    "cv_fold"
                ].isin(
                    [
                        outer_fold,
                        inner_validation_fold,
                    ]
                ),
                "patient_id",
            ]
            .astype(
                int
            )
            .tolist()
        )


        outer_inner_validation_overlap = len(
            outer_patients
            &
            inner_validation_patients
        )


        outer_inner_training_overlap = len(
            outer_patients
            &
            inner_training_patients
        )


        inner_training_validation_overlap = len(
            inner_training_patients
            &
            inner_validation_patients
        )


        if (
            outer_inner_validation_overlap
            >
            0
            or
            outer_inner_training_overlap
            >
            0
            or
            inner_training_validation_overlap
            >
            0
        ):

            raise RuntimeError(
                "Nested patient separation failed."
            )


        partition_audit_rows.append(
            {
                "outer_fold": (
                    outer_fold
                ),
                "inner_validation_fold": (
                    inner_validation_fold
                ),
                "n_inner_training_patients": len(
                    inner_training_patients
                ),
                "n_inner_validation_patients": len(
                    inner_validation_patients
                ),
                "n_outer_evaluation_patients": len(
                    outer_patients
                ),
                "outer_inner_training_overlap": (
                    outer_inner_training_overlap
                ),
                "outer_inner_validation_overlap": (
                    outer_inner_validation_overlap
                ),
                "inner_training_validation_overlap": (
                    inner_training_validation_overlap
                ),
            }
        )


partition_audit_df = pd.DataFrame(
    partition_audit_rows
)


if len(
    partition_audit_df
) != 20:

    raise RuntimeError(
        "Expected 20 nested partition combinations."
    )


atomic_csv_save(
    partition_audit_df,
    MANIFEST_ROOT
    / "nested_partition_separation_audit.csv",
    index=False,
)


# ==================================================================================================
# 13. Freeze / validate scientific analysis plan
# ==================================================================================================

UPSTREAM_FILES = {
    "development_manifest": (
        DEVELOPMENT_MANIFEST_PATH
    ),
    "patient_fold_assignment": (
        PATIENT_FOLD_ASSIGNMENT_PATH
    ),
    "patient_reference_audit": (
        PATIENT_REFERENCE_AUDIT_PATH
    ),
    "notebook06_analysis_plan": (
        NOTEBOOK06_PLAN_PATH
    ),
}


UPSTREAM_HASHES = {
    role: file_sha256(
        path
    )
    for role, path in UPSTREAM_FILES.items()
}


SCIENTIFIC_ANALYSIS_SPECIFICATION = {
    "protocol_version": PROTOCOL_VERSION,
    "supersedes": (
        "nested_patient_grouped_evaluation_v1_plan_only"
    ),
    "analysis_role": (
        "development-only genuine nested patient-grouped evaluation"
    ),
    "official_partition_access": False,
    "development_manifest_fingerprint": (
        DEVELOPMENT_MANIFEST_FINGERPRINT
    ),
    "outer_label_vault_fingerprint": (
        OUTER_LABEL_VAULT_FINGERPRINT
    ),
    "outer_design": {
        "n_outer_folds": (
            NUM_OUTER_FOLDS
        ),
        "outer_fold_ids": [
            0,
            1,
            2,
            3,
            4,
        ],
        "outer_fold_source": (
            "frozen development patient-fold assignment"
        ),
        "outer_prediction_policy": (
            "fold-level outer prediction files contain no true labels; "
            "labels are joined only after all five outer folds are complete"
        ),
        "outer_selection_use": False,
    },
    "inner_design": {
        "n_inner_folds_per_outer": (
            NUM_INNER_FOLDS_PER_OUTER
        ),
        "inner_fold_definition": (
            "the four frozen folds remaining after exclusion "
            "of the current outer fold"
        ),
        "screening_seed": (
            SCREENING_SEED
        ),
        "confirmation_seeds": (
            CONFIRMATION_SEEDS
        ),
        "screen_all_five_loss_candidates_per_family": True,
        "shortlist_size_per_family": 2,
        "checkpoint_aggregation_rule": (
            CHECKPOINT_AGGREGATION_RULE
        ),
    },
    "selection_sequence": [
        (
            "ViT five-loss screening with seed 42 using inner OOF "
            "mean-probability/modal-low performance"
        ),
        (
            "ViT top-two confirmation with seeds 42, 2025, and 2026"
        ),
        (
            "six-rule aggregation selection from the six confirmed "
            "ViT candidate-seed inner OOF sources"
        ),
        (
            "ViT configuration selection under the selected aggregation"
        ),
        (
            "Swin five-loss screening with seed 42 under the selected aggregation"
        ),
        (
            "Swin top-two confirmation with seeds 42, 2025, and 2026"
        ),
        (
            "Swin configuration selection under the selected aggregation"
        ),
        (
            "inner-only ViT-versus-Swin system selection"
        ),
        (
            "seed-specific outer-refit epoch derivation from inner best epochs"
        ),
        (
            "outer refit followed by label-blind outer inference"
        ),
    ],
    "candidate_models": (
        CANDIDATES
    ),
    "aggregation_candidates": (
        AGGREGATION_RULES
    ),
    "primary_reference": (
        PRIMARY_REFERENCE_RULE
    ),
    "candidate_and_system_ranking": [
        "higher patient QWK",
        "lower patient MAE",
        "higher patient macro-F1",
        "higher patient accuracy",
        "higher image QWK",
        "fixed deterministic candidate order",
    ],
    "aggregation_ranking": [
        "higher patient QWK",
        "lower patient MAE",
        "higher patient macro-F1",
        "higher patient accuracy",
        "fixed deterministic aggregation order",
    ],
    "outer_refit_epoch_rule": (
        "half-up-rounded median of the four inner-fold best epochs, "
        "computed separately for family, selected candidate, and seed"
    ),
    "outer_refit_scheduler_policy": (
        "original 25-epoch warm-up/cosine schedule truncated at "
        "the inner-selected refit epoch; schedule is not rescaled"
    ),
    "outer_outputs": [
        "nested ViT-family estimate",
        "nested Swin-family estimate",
        "fully nested selected-procedure estimate",
    ],
    "training_configuration": {
        "image_size": (
            IMAGE_SIZE
        ),
        "train_batch_size": (
            TRAIN_BATCH_SIZE
        ),
        "gradient_accumulation_steps": (
            GRADIENT_ACCUMULATION_STEPS
        ),
        "effective_train_batch_size": (
            EFFECTIVE_TRAIN_BATCH_SIZE
        ),
        "validation_batch_size": (
            VALIDATION_BATCH_SIZE
        ),
        "maximum_epochs": (
            MAX_EPOCHS
        ),
        "early_stopping_patience": (
            EARLY_STOP_PATIENCE
        ),
        "optimizer": (
            "AdamW"
        ),
        "initial_learning_rate": (
            LEARNING_RATE
        ),
        "weight_decay": (
            WEIGHT_DECAY
        ),
        "warmup_epochs": (
            WARMUP_EPOCHS
        ),
        "minimum_lr_ratio": (
            MIN_LR_RATIO
        ),
        "label_smoothing": (
            LABEL_SMOOTHING
        ),
        "drop_rate": (
            DROP_RATE
        ),
        "drop_path_rate": (
            DROP_PATH_RATE
        ),
        "gradient_clip_norm": (
            GRADIENT_CLIP_NORM
        ),
        "mixed_precision": (
            USE_MIXED_PRECISION
        ),
        "validation_tta": (
            "mean probabilities from original and horizontal-flip views"
        ),
        "image_class_weights": (
            USE_IMAGE_CLASS_WEIGHTS
        ),
    },
    "bootstrap": {
        "replicates": (
            BOOTSTRAP_REPLICATES
        ),
        "seed": (
            BOOTSTRAP_SEED
        ),
        "sampling_unit": (
            "patient"
        ),
        "stratification": (
            PRIMARY_REFERENCE_RULE
        ),
        "interval": (
            "2.5th and 97.5th percentiles"
        ),
    },
    "upstream_hashes": (
        UPSTREAM_HASHES
    ),
    "guardrails": [
        "No official-partition path is defined",
        "No official-partition file is loaded",
        "No outer patient contributes to inner training",
        "No outer patient contributes to inner validation",
        "No outer label contributes to checkpoint selection",
        "No outer label contributes to loss selection",
        "No outer label contributes to aggregation selection",
        "No outer label contributes to system selection",
        "No outer label contributes to refit-epoch selection",
        "Partial outer-fold runs generate no performance summaries",
        (
            "Nested results cannot modify the historical locked "
            "official-partition analysis"
        ),
    ],
}


ANALYSIS_PLAN_FINGERPRINT = (
    dictionary_sha256(
        SCIENTIFIC_ANALYSIS_SPECIFICATION
    )
)


EXPECTED_FROZEN_ANALYSIS_PLAN_FINGERPRINT = "5c5cab052235f11e10a43f0fc23cdc8831a89b502bcbefa85973ad6e5d076cd6"
if ANALYSIS_PLAN_FINGERPRINT != EXPECTED_FROZEN_ANALYSIS_PLAN_FINGERPRINT:
    raise RuntimeError("Computed analysis-plan fingerprint differs from this copy's frozen protocol.")


ANALYSIS_PLAN_PATH = (
    V2_ROOT
    / "analysis_plan_frozen_before_nested_evaluation_v2.json"
)


if ANALYSIS_PLAN_PATH.exists():

    with open(
        ANALYSIS_PLAN_PATH,
        "r",
        encoding="utf-8",
    ) as file:

        existing_plan = json.load(
            file
        )


    existing_fingerprint = (
        existing_plan.get(
            "analysis_plan_fingerprint"
        )
    )


    if (
        existing_fingerprint
        !=
        ANALYSIS_PLAN_FINGERPRINT
    ):

        raise RuntimeError(
            "\nSCIENTIFIC PLAN FINGERPRINT MISMATCH.\n\n"
            f"Existing:\n{existing_fingerprint}\n\n"
            f"Current:\n{ANALYSIS_PLAN_FINGERPRINT}\n\n"
            "Do not continue. Use the previously validated v2 code "
            "or create a new versioned protocol."
        )


    print(
        "\nExisting v2 plan validated:"
    )

    print(
        ANALYSIS_PLAN_PATH
    )


else:

    atomic_json_save(
        {
            "created_utc": (
                utc_now()
            ),
            "analysis_plan_fingerprint": (
                ANALYSIS_PLAN_FINGERPRINT
            ),
            "scientific_analysis_specification": (
                SCIENTIFIC_ANALYSIS_SPECIFICATION
            ),
        },
        ANALYSIS_PLAN_PATH,
    )


print(
    "\nV2 nested-analysis fingerprint:"
)

print(
    ANALYSIS_PLAN_FINGERPRINT
)


# ==================================================================================================
# 14. Environment / deterministic setup
# ==================================================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else
    "cpu"
)


if (
    EXECUTION_MODE
    ==
    "RUN"
    and
    DEVICE.type
    !=
    "cuda"
):

    raise RuntimeError(
        "Training requires CUDA."
    )


def seed_everything(
    seed,
):

    seed = int(
        seed
    )

    os.environ[
        "PYTHONHASHSEED"
    ] = str(
        seed
    )

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed(
            seed
        )

        torch.cuda.manual_seed_all(
            seed
        )


def seed_worker(
    worker_id,
):

    worker_seed = (
        torch.initial_seed()
        %
        (2**32)
    )

    np.random.seed(
        worker_seed
    )

    random.seed(
        worker_seed
    )


def as_cpu_byte_tensor(
    state,
):

    if isinstance(
        state,
        torch.Tensor,
    ):

        return (
            state.detach()
            .to(
                "cpu",
                dtype=torch.uint8,
            )
            .contiguous()
        )


    if isinstance(
        state,
        np.ndarray,
    ):

        return (
            torch.from_numpy(
                state.astype(
                    np.uint8,
                    copy=False,
                )
            )
            .cpu()
            .contiguous()
        )


    return torch.tensor(
        state,
        dtype=torch.uint8,
        device="cpu",
    ).contiguous()


def capture_rng_state():

    state = {
        "python": (
            random.getstate()
        ),
        "numpy": (
            np.random.get_state()
        ),
        "torch_cpu": (
            torch.get_rng_state()
            .cpu()
            .contiguous()
        ),
        "torch_cuda": None,
    }


    if torch.cuda.is_available():

        state[
            "torch_cuda"
        ] = [
            cuda_state.cpu()
            .contiguous()
            for cuda_state in (
                torch.cuda.get_rng_state_all()
            )
        ]


    return state


def restore_rng_state(
    state,
):

    random.setstate(
        state[
            "python"
        ]
    )


    numpy_state = (
        state[
            "numpy"
        ]
    )


    if isinstance(
        numpy_state,
        list,
    ):

        numpy_state = tuple(
            numpy_state
        )


    np.random.set_state(
        numpy_state
    )


    torch.set_rng_state(
        as_cpu_byte_tensor(
            state[
                "torch_cpu"
            ]
        )
    )


    if (
        torch.cuda.is_available()
        and
        state.get(
            "torch_cuda"
        )
        is not None
    ):

        cuda_states = [
            as_cpu_byte_tensor(
                cuda_state
            )
            for cuda_state in (
                state[
                    "torch_cuda"
                ]
            )
        ]


        if (
            len(
                cuda_states
            )
            !=
            torch.cuda.device_count()
        ):

            if (
                torch.cuda.device_count()
                ==
                1
                and
                len(
                    cuda_states
                )
                >=
                1
            ):

                cuda_states = [
                    cuda_states[
                        0
                    ]
                ]


            else:

                raise RuntimeError(
                    "CUDA RNG-state count mismatch."
                )


        torch.cuda.set_rng_state_all(
            cuda_states
        )


def optimizer_to_device(
    optimizer,
    target_device,
):

    for optimizer_state in (
        optimizer.state.values()
    ):

        for key, value in list(
            optimizer_state.items()
        ):

            if isinstance(
                value,
                torch.Tensor,
            ):

                optimizer_state[
                    key
                ] = value.to(
                    target_device
                )


seed_everything(
    SCREENING_SEED
)


torch.backends.cudnn.deterministic = True

torch.backends.cudnn.benchmark = False


try:

    torch.set_float32_matmul_precision(
        "high"
    )

except Exception:

    pass


print(
    "\nDevice:",
    DEVICE,
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(
            0
        ),
    )


# ==================================================================================================
# 15. Image I/O / cache
# ==================================================================================================

def cache_path_for_relative_path(
    relative_path,
):

    return (
        LOCAL_IMAGE_CACHE_ROOT
        /
        Path(
            str(
                relative_path
            )
        )
    )


def read_drive_bytes(
    source_path,
):

    source_path = Path(
        source_path
    )

    last_error = None


    for attempt_index in range(
        IMAGE_IO_MAX_RETRIES
    ):

        try:

            with open(
                source_path,
                "rb",
            ) as file:

                image_bytes = (
                    file.read()
                )


            if len(
                image_bytes
            ) == 0:

                raise OSError(
                    "Zero-byte image read."
                )


            return image_bytes


        except (
            OSError,
            IOError,
        ) as error:

            last_error = error


            if (
                attempt_index
                ==
                IMAGE_IO_MAX_RETRIES
                -
                1
            ):

                break


            delay = retry_delay(
                attempt_index,
                IMAGE_IO_INITIAL_DELAY_SECONDS,
                IMAGE_IO_MAX_DELAY_SECONDS,
            )


            print(
                "\nTransient image-read failure:",
                source_path,
            )


            print(
                "Retrying in",
                delay,
                "seconds."
            )


            time.sleep(
                delay
            )


    raise OSError(
        f"Unable to read image:\n{source_path}\n{repr(last_error)}"
    )


def ensure_cached_image(
    source_path,
    relative_path,
):

    local_path = (
        cache_path_for_relative_path(
            relative_path
        )
    )


    if (
        local_path.exists()
        and
        local_path.stat().st_size
        >
        0
    ):

        return local_path


    local_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    image_bytes = read_drive_bytes(
        source_path
    )


    temp_path = temporary_path(
        local_path
    )


    with open(
        temp_path,
        "wb",
    ) as file:

        file.write(
            image_bytes
        )


    os.replace(
        temp_path,
        local_path,
    )


    return local_path


def load_rgb_image(
    source_path,
    relative_path,
):

    local_path = None

    last_error = None


    for attempt_index in range(
        IMAGE_IO_MAX_RETRIES
    ):

        try:

            if USE_LOCAL_IMAGE_CACHE:

                local_path = ensure_cached_image(
                    source_path,
                    relative_path,
                )

                read_path = (
                    local_path
                )


            else:

                read_path = Path(
                    source_path
                )


            with open(
                read_path,
                "rb",
            ) as file:

                image_bytes = (
                    file.read()
                )


            with Image.open(
                io.BytesIO(
                    image_bytes
                )
            ) as image:

                image.load()

                return image.convert(
                    "RGB"
                )


        except (
            OSError,
            IOError,
            ValueError,
        ) as error:

            last_error = error


            if (
                local_path is not None
                and
                local_path.exists()
            ):

                try:
                    local_path.unlink()

                except Exception:
                    pass


            if (
                attempt_index
                ==
                IMAGE_IO_MAX_RETRIES
                -
                1
            ):

                break


            time.sleep(
                retry_delay(
                    attempt_index,
                    IMAGE_IO_INITIAL_DELAY_SECONDS,
                    IMAGE_IO_MAX_DELAY_SECONDS,
                )
            )


    raise OSError(
        f"Unable to decode image:\n{source_path}\n{repr(last_error)}"
    )


# ==================================================================================================
# 16. Dataset / transforms
# ==================================================================================================

class LabeledLIMUCDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform,
    ):

        self.dataframe = (
            dataframe.reset_index(
                drop=True
            )
            .copy()
        )

        self.transform = (
            transform
        )


    def __len__(
        self,
    ):

        return len(
            self.dataframe
        )


    def __getitem__(
        self,
        index,
    ):

        row = (
            self.dataframe.iloc[
                index
            ]
        )


        image = load_rgb_image(
            row[
                "path"
            ],
            row[
                "relative_path"
            ],
        )


        image = self.transform(
            image
        )


        return {
            "image": (
                image
            ),
            "label": torch.tensor(
                int(
                    row[
                        "label"
                    ]
                ),
                dtype=torch.long,
            ),
            "patient_id": torch.tensor(
                int(
                    row[
                        "patient_id"
                    ]
                ),
                dtype=torch.long,
            ),
            "patient_y_true": torch.tensor(
                int(
                    row[
                        "patient_y_true"
                    ]
                ),
                dtype=torch.long,
            ),
            "cv_fold": torch.tensor(
                int(
                    row[
                        "cv_fold"
                    ]
                ),
                dtype=torch.long,
            ),
            "path": str(
                row[
                    "path"
                ]
            ),
            "relative_path": str(
                row[
                    "relative_path"
                ]
            ),
        }


class BlindOuterInferenceDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform,
    ):

        forbidden_columns = {
            "label",
            "y_true",
            "patient_y_true",
        }


        present_forbidden = (
            forbidden_columns
            &
            set(
                dataframe.columns
            )
        )


        if present_forbidden:

            raise RuntimeError(
                "Blind outer inference dataframe contains labels:\n"
                f"{sorted(present_forbidden)}"
            )


        self.dataframe = (
            dataframe.reset_index(
                drop=True
            )
            .copy()
        )


        self.transform = (
            transform
        )


    def __len__(
        self,
    ):

        return len(
            self.dataframe
        )


    def __getitem__(
        self,
        index,
    ):

        row = (
            self.dataframe.iloc[
                index
            ]
        )


        image = load_rgb_image(
            row[
                "path"
            ],
            row[
                "relative_path"
            ],
        )


        image = self.transform(
            image
        )


        return {
            "image": (
                image
            ),
            "patient_id": torch.tensor(
                int(
                    row[
                        "patient_id"
                    ]
                ),
                dtype=torch.long,
            ),
            "cv_fold": torch.tensor(
                int(
                    row[
                        "cv_fold"
                    ]
                ),
                dtype=torch.long,
            ),
            "path": str(
                row[
                    "path"
                ]
            ),
            "relative_path": str(
                row[
                    "relative_path"
                ]
            ),
        }


def build_train_transform():

    return transforms.Compose(
        [
            transforms.RandomResizedCrop(
                (
                    IMAGE_SIZE,
                    IMAGE_SIZE,
                ),
                scale=(
                    0.75,
                    1.00,
                ),
                ratio=(
                    0.90,
                    1.10,
                ),
            ),
            transforms.RandomHorizontalFlip(
                p=0.5
            ),
            transforms.RandomRotation(
                degrees=8
            ),
            transforms.ColorJitter(
                brightness=0.12,
                contrast=0.12,
                saturation=0.08,
                hue=0.02,
            ),
            transforms.GaussianBlur(
                kernel_size=3,
                sigma=(
                    0.1,
                    1.2,
                ),
            ),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[
                    0.485,
                    0.456,
                    0.406,
                ],
                std=[
                    0.229,
                    0.224,
                    0.225,
                ],
            ),
        ]
    )


def build_eval_transform():

    return transforms.Compose(
        [
            transforms.Resize(
                (
                    IMAGE_SIZE,
                    IMAGE_SIZE,
                )
            ),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[
                    0.485,
                    0.456,
                    0.406,
                ],
                std=[
                    0.229,
                    0.224,
                    0.225,
                ],
            ),
        ]
    )


# ==================================================================================================
# 17. Metrics
# ==================================================================================================

def safe_qwk(
    y_true,
    y_pred,
):

    try:

        value = cohen_kappa_score(
            y_true,
            y_pred,
            labels=CLASS_IDS,
            weights="quadratic",
        )


        if not np.isfinite(
            value
        ):

            return np.nan


        return float(
            value
        )


    except Exception:

        return np.nan


def fixed_balanced_accuracy(
    y_true,
    y_pred,
):

    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=CLASS_IDS,
    ).astype(
        float
    )


    row_sums = matrix.sum(
        axis=1
    )


    recalls = np.divide(
        np.diag(
            matrix
        ),
        row_sums,
        out=np.zeros(
            NUM_CLASSES,
            dtype=float,
        ),
        where=(
            row_sums
            >
            0
        ),
    )


    present_classes = (
        row_sums
        >
        0
    )


    if not present_classes.any():

        return np.nan


    return float(
        recalls[
            present_classes
        ].mean()
    )


def compute_metrics(
    y_true,
    y_pred,
):

    y_true = np.asarray(
        y_true,
        dtype=int,
    )


    y_pred = np.asarray(
        y_pred,
        dtype=int,
    )


    return {
        "qwk": safe_qwk(
            y_true,
            y_pred,
        ),
        "mae": float(
            mean_absolute_error(
                y_true,
                y_pred,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=CLASS_IDS,
                average="macro",
                zero_division=0,
            )
        ),
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "balanced_accuracy": (
            fixed_balanced_accuracy(
                y_true,
                y_pred,
            )
        ),
    }


# ==================================================================================================
# 18. Aggregation
# ==================================================================================================

def normalize_probability_vector(
    vector,
):

    vector = np.asarray(
        vector,
        dtype=float,
    )


    vector = np.clip(
        vector,
        0.0,
        None,
    )


    total = float(
        vector.sum()
    )


    if total <= 0:

        return (
            np.ones(
                NUM_CLASSES,
                dtype=float,
            )
            /
            NUM_CLASSES
        )


    return (
        vector
        /
        total
    )


def round_half_up(
    value,
):

    return int(
        math.floor(
            float(
                value
            )
            +
            0.5
        )
    )


def round_half_up_to_class(
    value,
):

    return int(
        np.clip(
            round_half_up(
                value
            ),
            0,
            NUM_CLASSES
            -
            1,
        )
    )


def aggregate_one_patient(
    patient_group,
    aggregation_rule,
):

    probabilities = (
        patient_group[
            PROBABILITY_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
    )


    hard_predictions = np.argmax(
        probabilities,
        axis=1,
    )


    patient_truth_values = (
        patient_group[
            "patient_y_true"
        ]
        .astype(
            int
        )
        .unique()
    )


    if len(
        patient_truth_values
    ) != 1:

        raise RuntimeError(
            "A patient has inconsistent derived reference labels."
        )


    mean_probabilities = (
        normalize_probability_vector(
            probabilities.mean(
                axis=0
            )
        )
    )


    if (
        aggregation_rule
        ==
        "mean_probability_argmax"
    ):

        aggregate_probabilities = (
            mean_probabilities
        )

        patient_prediction = int(
            np.argmax(
                aggregate_probabilities
            )
        )


    elif (
        aggregation_rule
        ==
        "median_probability_argmax"
    ):

        aggregate_probabilities = (
            normalize_probability_vector(
                np.median(
                    probabilities,
                    axis=0,
                )
            )
        )


        patient_prediction = int(
            np.argmax(
                aggregate_probabilities
            )
        )


    elif (
        aggregation_rule
        ==
        "majority_vote_low_tie"
    ):

        vote_counts = np.bincount(
            hard_predictions,
            minlength=NUM_CLASSES,
        ).astype(
            float
        )


        tied_classes = np.flatnonzero(
            vote_counts
            ==
            vote_counts.max()
        )


        patient_prediction = int(
            tied_classes[
                0
            ]
        )


        aggregate_probabilities = (
            normalize_probability_vector(
                vote_counts
            )
        )


    elif (
        aggregation_rule
        ==
        "mean_expected_mes_round"
    ):

        expected_mes = (
            probabilities
            @
            np.arange(
                NUM_CLASSES,
                dtype=float,
            )
        )


        patient_prediction = (
            round_half_up_to_class(
                expected_mes.mean()
            )
        )


        aggregate_probabilities = np.zeros(
            NUM_CLASSES,
            dtype=float,
        )


        aggregate_probabilities[
            patient_prediction
        ] = 1.0


    elif (
        aggregation_rule
        ==
        "median_expected_mes_round"
    ):

        expected_mes = (
            probabilities
            @
            np.arange(
                NUM_CLASSES,
                dtype=float,
            )
        )


        patient_prediction = (
            round_half_up_to_class(
                np.median(
                    expected_mes
                )
            )
        )


        aggregate_probabilities = np.zeros(
            NUM_CLASSES,
            dtype=float,
        )


        aggregate_probabilities[
            patient_prediction
        ] = 1.0


    elif (
        aggregation_rule
        ==
        "maximum_predicted_image_class"
    ):

        patient_prediction = int(
            hard_predictions.max()
        )


        aggregate_probabilities = np.zeros(
            NUM_CLASSES,
            dtype=float,
        )


        aggregate_probabilities[
            patient_prediction
        ] = 1.0


    else:

        raise ValueError(
            f"Unknown aggregation rule: {aggregation_rule}"
        )


    output_row = {
        "patient_id": int(
            patient_group[
                "patient_id"
            ].iloc[
                0
            ]
        ),
        "y_true": int(
            patient_truth_values[
                0
            ]
        ),
        "y_pred": int(
            patient_prediction
        ),
        "n_images": int(
            len(
                patient_group
            )
        ),
        "aggregation_rule": (
            aggregation_rule
        ),
    }


    if "outer_fold" in patient_group.columns:

        values = (
            patient_group[
                "outer_fold"
            ]
            .dropna()
            .astype(
                int
            )
            .unique()
        )


        if len(
            values
        ) == 1:

            output_row[
                "outer_fold"
            ] = int(
                values[
                    0
                ]
            )


    for class_id, probability_column in enumerate(
        PROBABILITY_COLUMNS
    ):

        output_row[
            probability_column
        ] = float(
            aggregate_probabilities[
                class_id
            ]
        )


    return output_row


def aggregate_image_predictions(
    image_dataframe,
    aggregation_rule,
):

    rows = []


    for _, patient_group in (
        image_dataframe.groupby(
            "patient_id",
            sort=True,
        )
    ):

        rows.append(
            aggregate_one_patient(
                patient_group,
                aggregation_rule,
            )
        )


    return (
        pd.DataFrame(
            rows
        )
        .sort_values(
            "patient_id"
        )
        .reset_index(
            drop=True
        )
    )


# ==================================================================================================
# 19. Loss functions
# ==================================================================================================

class ExpectedValueOrdinalMAELoss(
    nn.Module
):

    def __init__(
        self,
        num_classes,
    ):

        super().__init__()


        self.register_buffer(
            "class_values",
            torch.arange(
                num_classes,
                dtype=torch.float32,
            ),
        )


    def forward(
        self,
        logits,
        targets,
    ):

        probabilities = torch.softmax(
            logits,
            dim=1,
        )


        expected_values = (
            probabilities
            *
            self.class_values.unsqueeze(
                0
            )
        ).sum(
            dim=1
        )


        return torch.abs(
            expected_values
            -
            targets.float()
        ).mean()


class ExpectedOrdinalDistanceLoss(
    nn.Module
):

    def __init__(
        self,
        num_classes,
        alpha,
    ):

        super().__init__()


        self.alpha = float(
            alpha
        )


        self.register_buffer(
            "class_values",
            torch.arange(
                num_classes,
                dtype=torch.float32,
            ),
        )


    def forward(
        self,
        logits,
        targets,
    ):

        probabilities = torch.softmax(
            logits,
            dim=1,
        )


        distances = torch.abs(
            targets.float()
            .unsqueeze(
                1
            )
            -
            self.class_values.unsqueeze(
                0
            )
        ) ** self.alpha


        return (
            probabilities
            *
            distances
        ).sum(
            dim=1
        ).mean()


class CombinedNestedLoss(
    nn.Module
):

    def __init__(
        self,
        class_weights,
        ordinal_kind,
        ordinal_alpha,
        ordinal_weight,
    ):

        super().__init__()


        self.cross_entropy = (
            nn.CrossEntropyLoss(
                weight=(
                    class_weights
                ),
                label_smoothing=(
                    LABEL_SMOOTHING
                ),
            )
        )


        self.ordinal_kind = str(
            ordinal_kind
        )


        self.ordinal_weight = float(
            ordinal_weight
        )


        if (
            self.ordinal_kind
            ==
            "none"
        ):

            self.ordinal_loss = None


        elif (
            self.ordinal_kind
            ==
            "expected_value_mae"
        ):

            self.ordinal_loss = (
                ExpectedValueOrdinalMAELoss(
                    NUM_CLASSES
                )
            )


        elif (
            self.ordinal_kind
            ==
            "expected_distance"
        ):

            self.ordinal_loss = (
                ExpectedOrdinalDistanceLoss(
                    NUM_CLASSES,
                    ordinal_alpha,
                )
            )


        else:

            raise ValueError(
                f"Unknown ordinal_kind: {ordinal_kind}"
            )


    def forward(
        self,
        logits,
        targets,
    ):

        cross_entropy = (
            self.cross_entropy(
                logits,
                targets,
            )
        )


        if self.ordinal_loss is None:

            ordinal = logits.new_zeros(
                ()
            )


        else:

            ordinal = self.ordinal_loss(
                logits,
                targets,
            )


        total = (
            cross_entropy
            +
            self.ordinal_weight
            *
            ordinal
        )


        return (
            total,
            cross_entropy.detach(),
            ordinal.detach(),
        )


# ==================================================================================================
# 20. Scheduler / AMP / weights
# ==================================================================================================

def build_scheduler(
    optimizer,
):

    def learning_rate_multiplier(
        epoch,
    ):

        if epoch < WARMUP_EPOCHS:

            return (
                float(
                    epoch
                    +
                    1
                )
                /
                float(
                    WARMUP_EPOCHS
                )
            )


        progress = (
            float(
                epoch
                -
                WARMUP_EPOCHS
            )
            /
            float(
                max(
                    1,
                    MAX_EPOCHS
                    -
                    WARMUP_EPOCHS,
                )
            )
        )


        cosine_value = (
            0.5
            *
            (
                1.0
                +
                math.cos(
                    math.pi
                    *
                    progress
                )
            )
        )


        return max(
            MIN_LR_RATIO,
            cosine_value,
        )


    return (
        torch.optim.lr_scheduler.LambdaLR(
            optimizer,
            lr_lambda=(
                learning_rate_multiplier
            ),
        )
    )


def create_scaler():

    enabled = (
        USE_MIXED_PRECISION
        and
        torch.cuda.is_available()
    )


    try:

        return torch.amp.GradScaler(
            "cuda",
            enabled=enabled,
        )


    except Exception:

        return torch.cuda.amp.GradScaler(
            enabled=enabled
        )


def autocast_context(
    enabled,
):

    try:

        return torch.amp.autocast(
            device_type="cuda",
            enabled=enabled,
        )


    except Exception:

        return torch.cuda.amp.autocast(
            enabled=enabled
        )


def calculate_image_class_weights(
    training_dataframe,
    device,
):

    class_counts = (
        training_dataframe[
            "label"
        ]
        .value_counts()
        .reindex(
            CLASS_IDS,
            fill_value=0,
        )
        .sort_index()
    )


    if (
        class_counts == 0
    ).any():

        raise RuntimeError(
            "Training partition is missing an image class."
        )


    weights = (
        class_counts.sum()
        /
        (
            NUM_CLASSES
            *
            class_counts
        )
    )


    return torch.tensor(
        weights.to_numpy(
            dtype=np.float32
        ),
        dtype=torch.float32,
        device=device,
    )


# ==================================================================================================
# 21. Dataloaders
# ==================================================================================================

def build_training_loader(
    training_dataframe,
    run_seed,
):

    dataset = (
        LabeledLIMUCDataset(
            training_dataframe,
            transform=(
                build_train_transform()
            ),
        )
    )


    generator = torch.Generator()


    generator.manual_seed(
        int(
            run_seed
        )
        +
        101
    )


    loader = DataLoader(
        dataset,
        batch_size=(
            TRAIN_BATCH_SIZE
        ),
        shuffle=True,
        num_workers=(
            NUM_WORKERS
        ),
        pin_memory=(
            DEVICE.type
            ==
            "cuda"
        ),
        generator=(
            generator
        ),
        worker_init_fn=(
            seed_worker
        ),
        persistent_workers=False,
    )


    return (
        loader,
        generator,
    )


def build_labeled_evaluation_loader(
    evaluation_dataframe,
    run_seed,
):

    dataset = (
        LabeledLIMUCDataset(
            evaluation_dataframe,
            transform=(
                build_eval_transform()
            ),
        )
    )


    generator = torch.Generator()


    generator.manual_seed(
        int(
            run_seed
        )
        +
        202
    )


    loader = DataLoader(
        dataset,
        batch_size=(
            VALIDATION_BATCH_SIZE
        ),
        shuffle=False,
        num_workers=(
            NUM_WORKERS
        ),
        pin_memory=(
            DEVICE.type
            ==
            "cuda"
        ),
        generator=(
            generator
        ),
        worker_init_fn=(
            seed_worker
        ),
        persistent_workers=False,
    )


    return (
        loader,
        generator,
    )


def build_blind_outer_loader(
    blind_outer_dataframe,
    run_seed,
):

    dataset = (
        BlindOuterInferenceDataset(
            blind_outer_dataframe,
            transform=(
                build_eval_transform()
            ),
        )
    )


    generator = torch.Generator()


    generator.manual_seed(
        int(
            run_seed
        )
        +
        303
    )


    loader = DataLoader(
        dataset,
        batch_size=(
            VALIDATION_BATCH_SIZE
        ),
        shuffle=False,
        num_workers=(
            NUM_WORKERS
        ),
        pin_memory=(
            DEVICE.type
            ==
            "cuda"
        ),
        generator=(
            generator
        ),
        worker_init_fn=(
            seed_worker
        ),
        persistent_workers=False,
    )


    return (
        loader,
        generator,
    )


# ==================================================================================================
# 22. Train one epoch
# ==================================================================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device,
):

    model.train()


    total_loss = 0.0

    total_cross_entropy = 0.0

    total_ordinal = 0.0


    all_true = []

    all_predicted = []


    use_amp = (
        USE_MIXED_PRECISION
        and
        device.type
        ==
        "cuda"
    )


    optimizer.zero_grad(
        set_to_none=True
    )


    number_of_batches = len(
        loader
    )


    for batch_index, batch in enumerate(
        loader
    ):

        images = batch[
            "image"
        ].to(
            device,
            non_blocking=True,
        )


        labels = batch[
            "label"
        ].to(
            device,
            non_blocking=True,
        )


        group_start = (
            batch_index
            //
            GRADIENT_ACCUMULATION_STEPS
        ) * GRADIENT_ACCUMULATION_STEPS


        group_size = min(
            GRADIENT_ACCUMULATION_STEPS,
            number_of_batches
            -
            group_start,
        )


        with autocast_context(
            use_amp
        ):

            logits = model(
                images
            )


            (
                raw_loss,
                cross_entropy_loss,
                ordinal_loss,
            ) = criterion(
                logits,
                labels,
            )


            scaled_loss = (
                raw_loss
                /
                group_size
            )


        if use_amp:

            scaler.scale(
                scaled_loss
            ).backward()


        else:

            scaled_loss.backward()


        should_step = (
            (
                batch_index
                +
                1
            )
            %
            GRADIENT_ACCUMULATION_STEPS
            ==
            0
            or
            (
                batch_index
                +
                1
            )
            ==
            number_of_batches
        )


        if should_step:

            if use_amp:

                scaler.unscale_(
                    optimizer
                )


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRADIENT_CLIP_NORM,
            )


            if use_amp:

                scaler.step(
                    optimizer
                )

                scaler.update()


            else:

                optimizer.step()


            optimizer.zero_grad(
                set_to_none=True
            )


        batch_size = int(
            images.size(
                0
            )
        )


        total_loss += (
            float(
                raw_loss.item()
            )
            *
            batch_size
        )


        total_cross_entropy += (
            float(
                cross_entropy_loss.item()
            )
            *
            batch_size
        )


        total_ordinal += (
            float(
                ordinal_loss.item()
            )
            *
            batch_size
        )


        predictions = logits.argmax(
            dim=1
        )


        all_true.extend(
            labels.detach()
            .cpu()
            .tolist()
        )


        all_predicted.extend(
            predictions.detach()
            .cpu()
            .tolist()
        )


    metrics = compute_metrics(
        all_true,
        all_predicted,
    )


    denominator = len(
        loader.dataset
    )


    metrics[
        "loss"
    ] = (
        total_loss
        /
        denominator
    )


    metrics[
        "cross_entropy_loss"
    ] = (
        total_cross_entropy
        /
        denominator
    )


    metrics[
        "ordinal_loss"
    ] = (
        total_ordinal
        /
        denominator
    )


    return metrics


# ==================================================================================================
# 23. Prediction functions
# ==================================================================================================

@torch.no_grad()
def predict_labeled_images(
    model,
    loader,
    device,
    criterion=None,
):

    model.eval()


    total_loss = 0.0


    paths = []

    relative_paths = []

    patient_ids = []

    patient_truth = []

    image_truth = []

    source_folds = []

    probability_rows = []


    use_amp = (
        USE_MIXED_PRECISION
        and
        device.type
        ==
        "cuda"
    )


    for batch in loader:

        images = batch[
            "image"
        ].to(
            device,
            non_blocking=True,
        )


        labels = batch[
            "label"
        ].to(
            device,
            non_blocking=True,
        )


        with autocast_context(
            use_amp
        ):

            logits = model(
                images
            )


            if criterion is not None:

                loss, _, _ = criterion(
                    logits,
                    labels,
                )


                total_loss += (
                    float(
                        loss.item()
                    )
                    *
                    images.size(
                        0
                    )
                )


            probability_views = [
                torch.softmax(
                    logits,
                    dim=1,
                )
            ]


            if (
                USE_VALIDATION_TTA
                and
                TTA_HORIZONTAL_FLIP
            ):

                flipped_logits = model(
                    torch.flip(
                        images,
                        dims=[
                            3
                        ],
                    )
                )


                probability_views.append(
                    torch.softmax(
                        flipped_logits,
                        dim=1,
                    )
                )


            probabilities = torch.stack(
                probability_views,
                dim=0,
            ).mean(
                dim=0
            )


        paths.extend(
            list(
                batch[
                    "path"
                ]
            )
        )


        relative_paths.extend(
            list(
                batch[
                    "relative_path"
                ]
            )
        )


        patient_ids.extend(
            batch[
                "patient_id"
            ].cpu().tolist()
        )


        patient_truth.extend(
            batch[
                "patient_y_true"
            ].cpu().tolist()
        )


        image_truth.extend(
            labels.cpu().tolist()
        )


        source_folds.extend(
            batch[
                "cv_fold"
            ].cpu().tolist()
        )


        probability_rows.extend(
            probabilities.cpu()
            .numpy()
            .tolist()
        )


    output = pd.DataFrame(
        {
            "path": (
                paths
            ),
            "relative_path": (
                relative_paths
            ),
            "patient_id": (
                patient_ids
            ),
            "y_true": (
                image_truth
            ),
            "patient_y_true": (
                patient_truth
            ),
            "cv_fold": (
                source_folds
            ),
        }
    )


    output = pd.concat(
        [
            output,
            pd.DataFrame(
                probability_rows,
                columns=(
                    PROBABILITY_COLUMNS
                ),
            ),
        ],
        axis=1,
    )


    output[
        "y_pred"
    ] = np.argmax(
        output[
            PROBABILITY_COLUMNS
        ].to_numpy(
            dtype=float
        ),
        axis=1,
    ).astype(
        int
    )


    image_metrics = compute_metrics(
        output[
            "y_true"
        ],
        output[
            "y_pred"
        ],
    )


    if criterion is not None:

        image_metrics[
            "loss"
        ] = (
            total_loss
            /
            len(
                loader.dataset
            )
        )


    return (
        output,
        image_metrics,
    )


@torch.no_grad()
def predict_blind_outer_images(
    model,
    loader,
    device,
):

    model.eval()


    paths = []

    relative_paths = []

    patient_ids = []

    source_folds = []

    probability_rows = []


    use_amp = (
        USE_MIXED_PRECISION
        and
        device.type
        ==
        "cuda"
    )


    for batch in loader:

        images = batch[
            "image"
        ].to(
            device,
            non_blocking=True,
        )


        with autocast_context(
            use_amp
        ):

            logits = model(
                images
            )


            probability_views = [
                torch.softmax(
                    logits,
                    dim=1,
                )
            ]


            if (
                USE_VALIDATION_TTA
                and
                TTA_HORIZONTAL_FLIP
            ):

                flipped_logits = model(
                    torch.flip(
                        images,
                        dims=[
                            3
                        ],
                    )
                )


                probability_views.append(
                    torch.softmax(
                        flipped_logits,
                        dim=1,
                    )
                )


            probabilities = torch.stack(
                probability_views,
                dim=0,
            ).mean(
                dim=0
            )


        paths.extend(
            list(
                batch[
                    "path"
                ]
            )
        )


        relative_paths.extend(
            list(
                batch[
                    "relative_path"
                ]
            )
        )


        patient_ids.extend(
            batch[
                "patient_id"
            ].cpu().tolist()
        )


        source_folds.extend(
            batch[
                "cv_fold"
            ].cpu().tolist()
        )


        probability_rows.extend(
            probabilities.cpu()
            .numpy()
            .tolist()
        )


    output = pd.DataFrame(
        {
            "path": (
                paths
            ),
            "relative_path": (
                relative_paths
            ),
            "patient_id": (
                patient_ids
            ),
            "cv_fold": (
                source_folds
            ),
        }
    )


    output = pd.concat(
        [
            output,
            pd.DataFrame(
                probability_rows,
                columns=(
                    PROBABILITY_COLUMNS
                ),
            ),
        ],
        axis=1,
    )


    output[
        "y_pred"
    ] = np.argmax(
        output[
            PROBABILITY_COLUMNS
        ].to_numpy(
            dtype=float
        ),
        axis=1,
    ).astype(
        int
    )


    forbidden_columns = {
        "label",
        "y_true",
        "patient_y_true",
    }


    if forbidden_columns & set(
        output.columns
    ):

        raise RuntimeError(
            "Blind outer predictions unexpectedly contain labels."
        )


    return output


# ==================================================================================================
# 24. Inner validation
# ==================================================================================================

def evaluate_inner_validation(
    model,
    validation_loader,
    criterion,
    device,
):

    (
        image_dataframe,
        image_metrics,
    ) = predict_labeled_images(
        model,
        validation_loader,
        device,
        criterion=criterion,
    )


    patient_dataframe = (
        aggregate_image_predictions(
            image_dataframe,
            CHECKPOINT_AGGREGATION_RULE,
        )
    )


    patient_metrics = compute_metrics(
        patient_dataframe[
            "y_true"
        ],
        patient_dataframe[
            "y_pred"
        ],
    )


    return (
        image_metrics,
        patient_metrics,
        image_dataframe,
        patient_dataframe,
    )


# ==================================================================================================
# 25. Checkpoint hierarchy
# ==================================================================================================

def finite_for_maximum(
    value,
):

    if (
        value is None
        or
        not np.isfinite(
            value
        )
    ):

        return -np.inf


    return float(
        value
    )


def finite_for_minimum(
    value,
):

    if (
        value is None
        or
        not np.isfinite(
            value
        )
    ):

        return np.inf


    return float(
        value
    )


def checkpoint_tuple(
    patient_metrics,
    image_metrics,
):

    return (
        finite_for_maximum(
            patient_metrics[
                "qwk"
            ]
        ),
        -finite_for_minimum(
            patient_metrics[
                "mae"
            ]
        ),
        finite_for_maximum(
            patient_metrics[
                "macro_f1"
            ]
        ),
        finite_for_maximum(
            image_metrics[
                "qwk"
            ]
        ),
    )


def checkpoint_is_better(
    candidate_patient_metrics,
    best_patient_metrics,
    candidate_image_metrics,
    best_image_metrics,
):

    if best_patient_metrics is None:

        return True


    return (
        checkpoint_tuple(
            candidate_patient_metrics,
            candidate_image_metrics,
        )
        >
        checkpoint_tuple(
            best_patient_metrics,
            best_image_metrics,
        )
    )


# ==================================================================================================
# 26. Seeds
# ==================================================================================================

def inner_run_seed(
    nominal_seed,
    outer_fold,
    inner_validation_fold,
):

    return int(
        nominal_seed
        +
        100000
        *
        outer_fold
        +
        1000
        *
        inner_validation_fold
    )


def outer_refit_run_seed(
    nominal_seed,
    outer_fold,
):

    return int(
        nominal_seed
        +
        1000000
        *
        outer_fold
        +
        777
    )


# ==================================================================================================
# 27. Inner training
# ==================================================================================================

def run_inner_training(
    outer_fold,
    inner_validation_fold,
    candidate,
    nominal_seed,
):

    candidate_id = (
        candidate[
            "candidate_id"
        ]
    )


    run_seed = inner_run_seed(
        nominal_seed,
        outer_fold,
        inner_validation_fold,
    )


    training_dataframe = (
        development_df.loc[
            ~development_df[
                "cv_fold"
            ].isin(
                [
                    outer_fold,
                    inner_validation_fold,
                ]
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    validation_dataframe = (
        development_df.loc[
            development_df[
                "cv_fold"
            ]
            ==
            inner_validation_fold
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    training_patients = set(
        training_dataframe[
            "patient_id"
        ]
    )


    validation_patients = set(
        validation_dataframe[
            "patient_id"
        ]
    )


    outer_patients = set(
        patient_manifest_df.loc[
            patient_manifest_df[
                "cv_fold"
            ]
            ==
            outer_fold,
            "patient_id",
        ]
    )


    if training_patients & validation_patients:

        raise RuntimeError(
            "Inner train/validation overlap."
        )


    if training_patients & outer_patients:

        raise RuntimeError(
            "Outer patient leaked into inner training."
        )


    if validation_patients & outer_patients:

        raise RuntimeError(
            "Outer patient leaked into inner validation."
        )


    run_directory = (
        INNER_ROOT
        /
        f"outer_fold_{outer_fold}"
        /
        candidate[
            "family"
        ]
        /
        candidate_id
        /
        f"seed_{nominal_seed}"
        /
        f"inner_validation_fold_{inner_validation_fold}"
    )


    run_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


    completed_path = (
        run_directory
        /
        "RUN_COMPLETED.json"
    )


    history_path = (
        run_directory
        /
        "history.csv"
    )


    image_prediction_path = (
        run_directory
        /
        "validation_image_predictions.csv"
    )


    patient_prediction_path = (
        run_directory
        /
        "validation_patient_predictions_mean_probability.csv"
    )


    best_checkpoint_path = (
        run_directory
        /
        "best_checkpoint.pt"
    )


    last_checkpoint_path = (
        run_directory
        /
        "last_epoch_checkpoint.pt"
    )


    run_configuration = {
        "analysis_plan_fingerprint": (
            ANALYSIS_PLAN_FINGERPRINT
        ),
        "outer_fold": (
            outer_fold
        ),
        "inner_validation_fold": (
            inner_validation_fold
        ),
        "candidate": (
            candidate
        ),
        "nominal_seed": (
            nominal_seed
        ),
        "run_seed": (
            run_seed
        ),
    }


    run_fingerprint = (
        dictionary_sha256(
            run_configuration
        )
    )


    if (
        completed_path.exists()
        and
        image_prediction_path.exists()
        and
        patient_prediction_path.exists()
    ):

        completed_record = json.loads(
            completed_path.read_text(
                encoding="utf-8"
            )
        )


        if (
            completed_record[
                "run_fingerprint"
            ]
            !=
            run_fingerprint
        ):

            raise RuntimeError(
                "Inner run fingerprint mismatch."
            )


        return {
            "completed_record": (
                completed_record
            ),
            "image_predictions": (
                pd.read_csv(
                    image_prediction_path
                )
            ),
            "patient_predictions": (
                pd.read_csv(
                    patient_prediction_path
                )
            ),
        }


    seed_everything(
        run_seed
    )


    training_loader, training_generator = (
        build_training_loader(
            training_dataframe,
            run_seed,
        )
    )


    validation_loader, validation_generator = (
        build_labeled_evaluation_loader(
            validation_dataframe,
            run_seed,
        )
    )


    resume_payload = None


    if last_checkpoint_path.exists():

        resume_payload = safe_torch_load(
            last_checkpoint_path,
            map_location="cpu",
        )


        if (
            resume_payload[
                "run_fingerprint"
            ]
            !=
            run_fingerprint
        ):

            raise RuntimeError(
                "Resume fingerprint mismatch."
            )


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    model = timm.create_model(
        candidate[
            "model_name"
        ],
        pretrained=(
            resume_payload
            is
            None
        ),
        num_classes=(
            NUM_CLASSES
        ),
        drop_rate=(
            DROP_RATE
        ),
        drop_path_rate=(
            DROP_PATH_RATE
        ),
    ).to(
        DEVICE
    )


    class_weights = (
        calculate_image_class_weights(
            training_dataframe,
            DEVICE,
        )
    )


    criterion = CombinedNestedLoss(
        class_weights=(
            class_weights
        ),
        ordinal_kind=(
            candidate[
                "ordinal_kind"
            ]
        ),
        ordinal_alpha=(
            candidate[
                "ordinal_alpha"
            ]
        ),
        ordinal_weight=(
            candidate[
                "ordinal_weight"
            ]
        ),
    ).to(
        DEVICE
    )


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=(
            LEARNING_RATE
        ),
        weight_decay=(
            WEIGHT_DECAY
        ),
    )


    scheduler = build_scheduler(
        optimizer
    )


    scaler = create_scaler()


    start_epoch = 1

    best_epoch = -1

    best_patient_metrics = None

    best_image_metrics = None

    patience_counter = 0

    history = []


    if resume_payload is not None:

        model.load_state_dict(
            resume_payload[
                "model_state_dict"
            ]
        )


        optimizer.load_state_dict(
            resume_payload[
                "optimizer_state_dict"
            ]
        )


        optimizer_to_device(
            optimizer,
            DEVICE,
        )


        scheduler.load_state_dict(
            resume_payload[
                "scheduler_state_dict"
            ]
        )


        scaler.load_state_dict(
            resume_payload[
                "scaler_state_dict"
            ]
        )


        start_epoch = (
            int(
                resume_payload[
                    "completed_epoch"
                ]
            )
            +
            1
        )


        best_epoch = int(
            resume_payload[
                "best_epoch"
            ]
        )


        best_patient_metrics = (
            resume_payload[
                "best_patient_metrics"
            ]
        )


        best_image_metrics = (
            resume_payload[
                "best_image_metrics"
            ]
        )


        patience_counter = int(
            resume_payload[
                "patience_counter"
            ]
        )


        history = list(
            resume_payload[
                "history"
            ]
        )


        training_generator.set_state(
            as_cpu_byte_tensor(
                resume_payload[
                    "training_generator_state"
                ]
            )
        )


        validation_generator.set_state(
            as_cpu_byte_tensor(
                resume_payload[
                    "validation_generator_state"
                ]
            )
        )


        restore_rng_state(
            resume_payload[
                "rng_state"
            ]
        )


    print(
        "\nINNER TRAINING |",
        "outer=",
        outer_fold,
        "| inner=",
        inner_validation_fold,
        "|",
        candidate_id,
        "| seed=",
        nominal_seed,
    )


    for epoch in range(
        start_epoch,
        MAX_EPOCHS
        +
        1,
    ):

        if (
            patience_counter
            >=
            EARLY_STOP_PATIENCE
        ):

            break


        training_metrics = train_one_epoch(
            model,
            training_loader,
            criterion,
            optimizer,
            scaler,
            DEVICE,
        )


        (
            validation_image_metrics,
            validation_patient_metrics,
            _,
            _,
        ) = evaluate_inner_validation(
            model,
            validation_loader,
            criterion,
            DEVICE,
        )


        improved = checkpoint_is_better(
            validation_patient_metrics,
            best_patient_metrics,
            validation_image_metrics,
            best_image_metrics,
        )


        if improved:

            best_epoch = epoch

            best_patient_metrics = copy.deepcopy(
                validation_patient_metrics
            )

            best_image_metrics = copy.deepcopy(
                validation_image_metrics
            )

            patience_counter = 0


            atomic_torch_save(
                {
                    "run_fingerprint": (
                        run_fingerprint
                    ),
                    "best_epoch": (
                        best_epoch
                    ),
                    "best_patient_metrics": (
                        best_patient_metrics
                    ),
                    "best_image_metrics": (
                        best_image_metrics
                    ),
                    "model_state_dict": {
                        key: value.detach()
                        .cpu()
                        for key, value in (
                            model.state_dict()
                            .items()
                        )
                    },
                },
                best_checkpoint_path,
            )


        else:

            patience_counter += 1


        history.append(
            {
                "epoch": (
                    epoch
                ),
                "patient_qwk": (
                    validation_patient_metrics[
                        "qwk"
                    ]
                ),
                "patient_mae": (
                    validation_patient_metrics[
                        "mae"
                    ]
                ),
                "patient_macro_f1": (
                    validation_patient_metrics[
                        "macro_f1"
                    ]
                ),
                "image_qwk": (
                    validation_image_metrics[
                        "qwk"
                    ]
                ),
                "training_loss": (
                    training_metrics[
                        "loss"
                    ]
                ),
                "best_epoch": (
                    best_epoch
                ),
                "patience_counter": (
                    patience_counter
                ),
            }
        )


        atomic_csv_save(
            pd.DataFrame(
                history
            ),
            history_path,
            index=False,
        )


        scheduler.step()


        atomic_torch_save(
            {
                "run_fingerprint": (
                    run_fingerprint
                ),
                "completed_epoch": (
                    epoch
                ),
                "best_epoch": (
                    best_epoch
                ),
                "best_patient_metrics": (
                    best_patient_metrics
                ),
                "best_image_metrics": (
                    best_image_metrics
                ),
                "patience_counter": (
                    patience_counter
                ),
                "history": (
                    history
                ),
                "model_state_dict": (
                    model.state_dict()
                ),
                "optimizer_state_dict": (
                    optimizer.state_dict()
                ),
                "scheduler_state_dict": (
                    scheduler.state_dict()
                ),
                "scaler_state_dict": (
                    scaler.state_dict()
                ),
                "rng_state": (
                    capture_rng_state()
                ),
                "training_generator_state": (
                    training_generator
                    .get_state()
                    .cpu()
                ),
                "validation_generator_state": (
                    validation_generator
                    .get_state()
                    .cpu()
                ),
            },
            last_checkpoint_path,
        )


        print(
            f"Epoch {epoch:02d} | "
            f"QWK={validation_patient_metrics['qwk']:.4f} | "
            f"MAE={validation_patient_metrics['mae']:.4f} | "
            f"F1={validation_patient_metrics['macro_f1']:.4f} | "
            f"best={best_epoch} | "
            f"patience={patience_counter}"
        )


    best_checkpoint = safe_torch_load(
        best_checkpoint_path,
        map_location="cpu",
    )


    model.load_state_dict(
        best_checkpoint[
            "model_state_dict"
        ]
    )


    (
        final_image_metrics,
        final_patient_metrics,
        final_image_dataframe,
        final_patient_dataframe,
    ) = evaluate_inner_validation(
        model,
        validation_loader,
        criterion,
        DEVICE,
    )


    final_image_dataframe[
        "outer_fold"
    ] = outer_fold


    final_image_dataframe[
        "inner_validation_fold"
    ] = inner_validation_fold


    final_image_dataframe[
        "candidate_id"
    ] = candidate_id


    final_image_dataframe[
        "family"
    ] = candidate[
        "family"
    ]


    final_image_dataframe[
        "nominal_seed"
    ] = nominal_seed


    final_patient_dataframe[
        "outer_fold"
    ] = outer_fold


    final_patient_dataframe[
        "inner_validation_fold"
    ] = inner_validation_fold


    final_patient_dataframe[
        "candidate_id"
    ] = candidate_id


    final_patient_dataframe[
        "family"
    ] = candidate[
        "family"
    ]


    final_patient_dataframe[
        "nominal_seed"
    ] = nominal_seed


    atomic_csv_save(
        final_image_dataframe,
        image_prediction_path,
        index=False,
    )


    atomic_csv_save(
        final_patient_dataframe,
        patient_prediction_path,
        index=False,
    )


    completed_record = {
        "completed_utc": (
            utc_now()
        ),
        "run_fingerprint": (
            run_fingerprint
        ),
        "best_epoch": int(
            best_checkpoint[
                "best_epoch"
            ]
        ),
        "final_image_metrics": (
            final_image_metrics
        ),
        "final_patient_metrics_mean_probability": (
            final_patient_metrics
        ),
        "official_partition_loaded": False,
    }


    atomic_json_save(
        completed_record,
        completed_path,
    )


    if not KEEP_COMPLETED_INNER_CHECKPOINTS:

        for path in [
            best_checkpoint_path,
            last_checkpoint_path,
        ]:

            try:

                if path.exists():

                    path.unlink()

            except Exception:

                pass


    del model

    del optimizer

    del scheduler

    del scaler

    del criterion

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return {
        "completed_record": (
            completed_record
        ),
        "image_predictions": (
            final_image_dataframe
        ),
        "patient_predictions": (
            final_patient_dataframe
        ),
    }


# ==================================================================================================
# 28. Build complete inner OOF source
# ==================================================================================================

def build_inner_candidate_seed_source(
    outer_fold,
    candidate,
    nominal_seed,
):

    inner_fold_ids = [
        fold_id
        for fold_id in range(
            NUM_OUTER_FOLDS
        )
        if fold_id
        !=
        outer_fold
    ]


    image_parts = []

    fold_summary_rows = []


    for inner_validation_fold in inner_fold_ids:

        output = run_inner_training(
            outer_fold,
            inner_validation_fold,
            candidate,
            nominal_seed,
        )


        image_parts.append(
            output[
                "image_predictions"
            ]
        )


        record = (
            output[
                "completed_record"
            ]
        )


        fold_summary_rows.append(
            {
                "outer_fold": (
                    outer_fold
                ),
                "inner_validation_fold": (
                    inner_validation_fold
                ),
                "candidate_id": (
                    candidate[
                        "candidate_id"
                    ]
                ),
                "family": (
                    candidate[
                        "family"
                    ]
                ),
                "nominal_seed": (
                    nominal_seed
                ),
                "best_epoch": (
                    record[
                        "best_epoch"
                    ]
                ),
            }
        )


    oof_image_dataframe = (
        pd.concat(
            image_parts,
            ignore_index=True,
        )
        .sort_values(
            [
                "patient_id",
                "relative_path",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    image_metrics = compute_metrics(
        oof_image_dataframe[
            "y_true"
        ],
        oof_image_dataframe[
            "y_pred"
        ],
    )


    metric_rows = []

    patient_parts = []


    for aggregation_rule in AGGREGATION_RULES:

        patient_dataframe = (
            aggregate_image_predictions(
                oof_image_dataframe,
                aggregation_rule,
            )
        )


        patient_metrics = compute_metrics(
            patient_dataframe[
                "y_true"
            ],
            patient_dataframe[
                "y_pred"
            ],
        )


        patient_dataframe[
            "aggregation_rule"
        ] = aggregation_rule


        patient_dataframe[
            "candidate_id"
        ] = candidate[
            "candidate_id"
        ]


        patient_dataframe[
            "nominal_seed"
        ] = nominal_seed


        patient_parts.append(
            patient_dataframe
        )


        metric_rows.append(
            {
                "outer_fold": (
                    outer_fold
                ),
                "family": (
                    candidate[
                        "family"
                    ]
                ),
                "candidate_id": (
                    candidate[
                        "candidate_id"
                    ]
                ),
                "loss_code": (
                    candidate[
                        "loss_code"
                    ]
                ),
                "loss_name": (
                    candidate[
                        "loss_name"
                    ]
                ),
                "ordinal_kind": (
                    candidate[
                        "ordinal_kind"
                    ]
                ),
                "ordinal_alpha": (
                    candidate[
                        "ordinal_alpha"
                    ]
                ),
                "ordinal_weight": (
                    candidate[
                        "ordinal_weight"
                    ]
                ),
                "nominal_seed": (
                    nominal_seed
                ),
                "aggregation_rule": (
                    aggregation_rule
                ),
                "aggregation_label": (
                    AGGREGATION_LABELS[
                        aggregation_rule
                    ]
                ),
                "patient_qwk": (
                    patient_metrics[
                        "qwk"
                    ]
                ),
                "patient_mae": (
                    patient_metrics[
                        "mae"
                    ]
                ),
                "patient_macro_f1": (
                    patient_metrics[
                        "macro_f1"
                    ]
                ),
                "patient_accuracy": (
                    patient_metrics[
                        "accuracy"
                    ]
                ),
                "patient_balanced_accuracy": (
                    patient_metrics[
                        "balanced_accuracy"
                    ]
                ),
                "image_qwk": (
                    image_metrics[
                        "qwk"
                    ]
                ),
                "image_mae": (
                    image_metrics[
                        "mae"
                    ]
                ),
            }
        )


    return {
        "oof_image_predictions": (
            oof_image_dataframe
        ),
        "patient_predictions_long": (
            pd.concat(
                patient_parts,
                ignore_index=True,
            )
        ),
        "metrics_by_aggregation": (
            pd.DataFrame(
                metric_rows
            )
        ),
        "fold_summary": (
            pd.DataFrame(
                fold_summary_rows
            )
        ),
    }


# ==================================================================================================
# 29. Ranking helpers
# ==================================================================================================

def summarize_candidate_metrics(
    metrics_dataframe,
    candidate_ids,
    aggregation_rule,
):

    subset = metrics_dataframe.loc[
        metrics_dataframe[
            "candidate_id"
        ].isin(
            candidate_ids
        )
        &
        (
            metrics_dataframe[
                "aggregation_rule"
            ]
            ==
            aggregation_rule
        )
    ]


    summary = (
        subset.groupby(
            [
                "family",
                "candidate_id",
                "loss_code",
                "loss_name",
                "ordinal_kind",
                "ordinal_alpha",
                "ordinal_weight",
            ],
            as_index=False,
        )
        .agg(
            patient_qwk_mean=(
                "patient_qwk",
                "mean",
            ),
            patient_qwk_std=(
                "patient_qwk",
                "std",
            ),
            patient_mae_mean=(
                "patient_mae",
                "mean",
            ),
            patient_macro_f1_mean=(
                "patient_macro_f1",
                "mean",
            ),
            patient_accuracy_mean=(
                "patient_accuracy",
                "mean",
            ),
            image_qwk_mean=(
                "image_qwk",
                "mean",
            ),
        )
    )


    summary[
        "_candidate_order"
    ] = summary[
        "candidate_id"
    ].map(
        CANDIDATE_ORDER
    )


    summary = (
        summary.sort_values(
            [
                "patient_qwk_mean",
                "patient_mae_mean",
                "patient_macro_f1_mean",
                "patient_accuracy_mean",
                "image_qwk_mean",
                "_candidate_order",
            ],
            ascending=[
                False,
                True,
                False,
                False,
                False,
                True,
            ],
        )
        .drop(
            columns=[
                "_candidate_order"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    return summary


def summarize_aggregation_metrics(
    metrics_dataframe,
    candidate_ids,
):

    subset = metrics_dataframe.loc[
        metrics_dataframe[
            "candidate_id"
        ].isin(
            candidate_ids
        )
    ]


    summary = (
        subset.groupby(
            [
                "aggregation_rule",
                "aggregation_label",
            ],
            as_index=False,
        )
        .agg(
            patient_qwk_mean=(
                "patient_qwk",
                "mean",
            ),
            patient_mae_mean=(
                "patient_mae",
                "mean",
            ),
            patient_macro_f1_mean=(
                "patient_macro_f1",
                "mean",
            ),
            patient_accuracy_mean=(
                "patient_accuracy",
                "mean",
            ),
        )
    )


    summary[
        "_order"
    ] = summary[
        "aggregation_rule"
    ].map(
        AGGREGATION_ORDER
    )


    summary = (
        summary.sort_values(
            [
                "patient_qwk_mean",
                "patient_mae_mean",
                "patient_macro_f1_mean",
                "patient_accuracy_mean",
                "_order",
            ],
            ascending=[
                False,
                True,
                False,
                False,
                True,
            ],
        )
        .drop(
            columns=[
                "_order"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    return summary


def rank_selected_systems(
    vit_row,
    swin_row,
):

    dataframe = pd.DataFrame(
        [
            {
                "family": (
                    "ViT"
                ),
                "candidate_id": (
                    vit_row[
                        "candidate_id"
                    ]
                ),
                "patient_qwk_mean": (
                    vit_row[
                        "patient_qwk_mean"
                    ]
                ),
                "patient_mae_mean": (
                    vit_row[
                        "patient_mae_mean"
                    ]
                ),
                "patient_macro_f1_mean": (
                    vit_row[
                        "patient_macro_f1_mean"
                    ]
                ),
                "patient_accuracy_mean": (
                    vit_row[
                        "patient_accuracy_mean"
                    ]
                ),
                "image_qwk_mean": (
                    vit_row[
                        "image_qwk_mean"
                    ]
                ),
            },
            {
                "family": (
                    "Swin"
                ),
                "candidate_id": (
                    swin_row[
                        "candidate_id"
                    ]
                ),
                "patient_qwk_mean": (
                    swin_row[
                        "patient_qwk_mean"
                    ]
                ),
                "patient_mae_mean": (
                    swin_row[
                        "patient_mae_mean"
                    ]
                ),
                "patient_macro_f1_mean": (
                    swin_row[
                        "patient_macro_f1_mean"
                    ]
                ),
                "patient_accuracy_mean": (
                    swin_row[
                        "patient_accuracy_mean"
                    ]
                ),
                "image_qwk_mean": (
                    swin_row[
                        "image_qwk_mean"
                    ]
                ),
            },
        ]
    )


    dataframe[
        "_system_order"
    ] = dataframe[
        "family"
    ].map(
        {
            "ViT": 0,
            "Swin": 1,
        }
    )


    dataframe = (
        dataframe.sort_values(
            [
                "patient_qwk_mean",
                "patient_mae_mean",
                "patient_macro_f1_mean",
                "patient_accuracy_mean",
                "image_qwk_mean",
                "_system_order",
            ],
            ascending=[
                False,
                True,
                False,
                False,
                False,
                True,
            ],
        )
        .drop(
            columns=[
                "_system_order"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    return dataframe


# ==================================================================================================
# 30. Inner selection for one outer fold
# ==================================================================================================

def derive_outer_refit_epoch(
    fold_summary_dataframe,
):

    best_epochs = (
        fold_summary_dataframe[
            "best_epoch"
        ]
        .astype(
            int
        )
        .to_numpy()
    )


    median_epoch = float(
        np.median(
            best_epochs
        )
    )


    refit_epoch = int(
        np.clip(
            round_half_up(
                median_epoch
            ),
            1,
            MAX_EPOCHS,
        )
    )


    return refit_epoch


def select_within_outer_training_set(
    outer_fold,
):

    selection_directory = (
        INNER_ROOT
        /
        f"outer_fold_{outer_fold}"
        /
        "selection_records"
    )


    selection_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


    selection_path = (
        selection_directory
        /
        "outer_fold_inner_selection_record.json"
    )


    if selection_path.exists():

        selection_record = json.loads(
            selection_path.read_text(
                encoding="utf-8"
            )
        )


        if (
            selection_record[
                "analysis_plan_fingerprint"
            ]
            !=
            ANALYSIS_PLAN_FINGERPRINT
        ):

            raise RuntimeError(
                "Selection-plan fingerprint mismatch."
            )


        return selection_record


    cache = {}


    def ensure(
        candidate_id,
        nominal_seed,
    ):

        key = (
            candidate_id,
            nominal_seed,
        )


        if key not in cache:

            cache[
                key
            ] = build_inner_candidate_seed_source(
                outer_fold,
                CANDIDATE_LOOKUP[
                    candidate_id
                ],
                nominal_seed,
            )


        return cache[
            key
        ]


    # ViT screening
    for candidate_id in VIT_CANDIDATE_IDS:

        ensure(
            candidate_id,
            SCREENING_SEED,
        )


    all_metrics = pd.concat(
        [
            result[
                "metrics_by_aggregation"
            ]
            for result in cache.values()
        ],
        ignore_index=True,
    )


    vit_screening = summarize_candidate_metrics(
        all_metrics,
        VIT_CANDIDATE_IDS,
        CHECKPOINT_AGGREGATION_RULE,
    )


    vit_shortlist = (
        vit_screening.head(
            2
        )[
            "candidate_id"
        ].tolist()
    )


    # ViT confirmation
    for candidate_id in vit_shortlist:

        for seed in CONFIRMATION_SEEDS:

            ensure(
                candidate_id,
                seed,
            )


    all_metrics = pd.concat(
        [
            result[
                "metrics_by_aggregation"
            ]
            for result in cache.values()
        ],
        ignore_index=True,
    )


    aggregation_ranking = (
        summarize_aggregation_metrics(
            all_metrics,
            vit_shortlist,
        )
    )


    selected_aggregation_rule = (
        aggregation_ranking.iloc[
            0
        ][
            "aggregation_rule"
        ]
    )


    vit_confirmation = (
        summarize_candidate_metrics(
            all_metrics,
            vit_shortlist,
            selected_aggregation_rule,
        )
    )


    selected_vit_candidate = (
        vit_confirmation.iloc[
            0
        ][
            "candidate_id"
        ]
    )


    # Swin screening
    for candidate_id in SWIN_CANDIDATE_IDS:

        ensure(
            candidate_id,
            SCREENING_SEED,
        )


    all_metrics = pd.concat(
        [
            result[
                "metrics_by_aggregation"
            ]
            for result in cache.values()
        ],
        ignore_index=True,
    )


    swin_screening = (
        summarize_candidate_metrics(
            all_metrics,
            SWIN_CANDIDATE_IDS,
            selected_aggregation_rule,
        )
    )


    swin_shortlist = (
        swin_screening.head(
            2
        )[
            "candidate_id"
        ].tolist()
    )


    for candidate_id in swin_shortlist:

        for seed in CONFIRMATION_SEEDS:

            ensure(
                candidate_id,
                seed,
            )


    all_metrics = pd.concat(
        [
            result[
                "metrics_by_aggregation"
            ]
            for result in cache.values()
        ],
        ignore_index=True,
    )


    swin_confirmation = (
        summarize_candidate_metrics(
            all_metrics,
            swin_shortlist,
            selected_aggregation_rule,
        )
    )


    selected_swin_candidate = (
        swin_confirmation.iloc[
            0
        ][
            "candidate_id"
        ]
    )


    system_ranking = rank_selected_systems(
        vit_confirmation.iloc[
            0
        ],
        swin_confirmation.iloc[
            0
        ],
    )


    selected_system = (
        system_ranking.iloc[
            0
        ][
            "family"
        ]
    )


    selected_candidate_by_family = {
        "ViT": (
            selected_vit_candidate
        ),
        "Swin": (
            selected_swin_candidate
        ),
    }


    refit_epochs = {
        "ViT": {},
        "Swin": {},
    }


    refit_epoch_rows = []


    for family_name, candidate_id in (
        selected_candidate_by_family.items()
    ):

        for seed in CONFIRMATION_SEEDS:

            result = ensure(
                candidate_id,
                seed,
            )


            epoch = derive_outer_refit_epoch(
                result[
                    "fold_summary"
                ]
            )


            refit_epochs[
                family_name
            ][
                str(
                    seed
                )
            ] = epoch


            refit_epoch_rows.append(
                {
                    "outer_fold": (
                        outer_fold
                    ),
                    "family": (
                        family_name
                    ),
                    "candidate_id": (
                        candidate_id
                    ),
                    "nominal_seed": (
                        seed
                    ),
                    "outer_refit_epochs": (
                        epoch
                    ),
                }
            )


    atomic_csv_save(
        pd.DataFrame(
            refit_epoch_rows
        ),
        selection_directory
        /
        "07_outer_refit_epoch_plan.csv",
        index=False,
    )


    selection_record = {
        "completed_utc": (
            utc_now()
        ),
        "analysis_plan_fingerprint": (
            ANALYSIS_PLAN_FINGERPRINT
        ),
        "outer_fold": (
            outer_fold
        ),
        "outer_fold_used_for_any_selection": False,
        "outer_labels_used_for_any_selection": False,
        "vit_shortlist": (
            vit_shortlist
        ),
        "selected_aggregation_rule": (
            selected_aggregation_rule
        ),
        "selected_vit_candidate_id": (
            selected_vit_candidate
        ),
        "swin_shortlist": (
            swin_shortlist
        ),
        "selected_swin_candidate_id": (
            selected_swin_candidate
        ),
        "selected_system": (
            selected_system
        ),
        "selected_candidate_by_family": (
            selected_candidate_by_family
        ),
        "refit_epochs_by_family_and_seed": (
            refit_epochs
        ),
        "official_partition_loaded": False,
    }


    selection_record[
        "selection_record_fingerprint"
    ] = dictionary_sha256(
        selection_record
    )


    atomic_json_save(
        selection_record,
        selection_path,
    )


    print(
        "\nCompleted inner selection for outer fold",
        outer_fold,
    )


    print(
        "Selected aggregation:",
        selected_aggregation_rule,
    )


    print(
        "Selected ViT:",
        selected_vit_candidate,
    )


    print(
        "Selected Swin:",
        selected_swin_candidate,
    )


    print(
        "Selected system:",
        selected_system,
    )


    return selection_record


# ==================================================================================================
# 31. Outer refit directory / completion
# ==================================================================================================

def outer_refit_directory(
    outer_fold,
    family_name,
    candidate_id,
    nominal_seed,
):

    return (
        OUTER_ROOT
        /
        f"outer_fold_{outer_fold}"
        /
        family_name
        /
        candidate_id
        /
        f"seed_{nominal_seed}"
    )


def outer_fold_completion_path(
    outer_fold,
):

    return (
        OUTER_ROOT
        /
        f"outer_fold_{outer_fold}"
        /
        "OUTER_FOLD_COMPLETED.json"
    )


def all_outer_folds_complete():

    return all(
        outer_fold_completion_path(
            fold
        ).exists()
        for fold in range(
            NUM_OUTER_FOLDS
        )
    )


def incomplete_outer_folds():

    return [
        fold
        for fold in range(
            NUM_OUTER_FOLDS
        )
        if not outer_fold_completion_path(
            fold
        ).exists()
    ]


# ==================================================================================================
# 32. Outer refit / blind inference
# ==================================================================================================

def validate_completed_outer_fold(outer_fold):
    """Validate a saved completion receipt before skipping or assembling its fold."""
    completion_path = outer_fold_completion_path(outer_fold)
    if not completion_path.exists():
        raise FileNotFoundError(f"Outer fold {outer_fold} is not complete: {completion_path}")
    completion = json.loads(completion_path.read_text(encoding="utf-8"))
    if completion.get("analysis_plan_fingerprint") != ANALYSIS_PLAN_FINGERPRINT:
        raise RuntimeError(f"Outer fold {outer_fold} completion fingerprint mismatch.")
    if int(completion.get("outer_fold", -1)) != outer_fold:
        raise RuntimeError(f"Outer fold {outer_fold} completion record has wrong fold.")
    if int(completion.get("n_completed_outer_refits", -1)) != 6:
        raise RuntimeError(f"Outer fold {outer_fold} does not report six outer refits.")
    if bool(completion.get("outer_predictions_contain_labels", True)):
        raise RuntimeError(f"Outer fold {outer_fold} does not confirm label-blind predictions.")


def validate_outer_checkpoint_fingerprint(resume_payload, run_fingerprint):
    """Fail closed before model, optimizer, scheduler, scaler or RNG restoration."""
    if not isinstance(resume_payload, dict) or resume_payload.get("run_fingerprint") != run_fingerprint:
        raise RuntimeError("Outer refit checkpoint fingerprint mismatch; refusing to resume.")


def run_outer_refit_blind(
    outer_fold,
    family_name,
    candidate_id,
    nominal_seed,
    refit_epochs,
    selected_aggregation_rule,
    selection_record_fingerprint,
):

    candidate = (
        CANDIDATE_LOOKUP[
            candidate_id
        ]
    )


    training_dataframe = (
        development_df.loc[
            development_df[
                "cv_fold"
            ]
            !=
            outer_fold
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    blind_outer_dataframe = (
        development_df.loc[
            development_df[
                "cv_fold"
            ]
            ==
            outer_fold,
            [
                "path",
                "relative_path",
                "patient_id",
                "cv_fold",
            ],
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    run_seed = outer_refit_run_seed(
        nominal_seed,
        outer_fold,
    )


    run_directory = (
        outer_refit_directory(
            outer_fold,
            family_name,
            candidate_id,
            nominal_seed,
        )
    )


    run_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


    completed_path = (
        run_directory
        /
        "RUN_COMPLETED.json"
    )


    history_path = (
        run_directory
        /
        "refit_training_history.csv"
    )


    blind_prediction_path = (
        run_directory
        /
        "outer_blind_image_predictions.csv"
    )


    last_checkpoint_path = (
        run_directory
        /
        "last_epoch_checkpoint.pt"
    )


    run_fingerprint = (
        dictionary_sha256(
            {
                "analysis_plan_fingerprint": (
                    ANALYSIS_PLAN_FINGERPRINT
                ),
                "selection_record_fingerprint": (
                    selection_record_fingerprint
                ),
                "outer_fold": (
                    outer_fold
                ),
                "family": (
                    family_name
                ),
                "candidate": (
                    candidate
                ),
                "nominal_seed": (
                    nominal_seed
                ),
                "run_seed": (
                    run_seed
                ),
                "refit_epochs": (
                    refit_epochs
                ),
                "aggregation": (
                    selected_aggregation_rule
                ),
            }
        )
    )


    if (
        completed_path.exists()
        and
        blind_prediction_path.exists()
    ):

        completed_record = json.loads(
            completed_path.read_text(
                encoding="utf-8"
            )
        )


        if (
            completed_record[
                "run_fingerprint"
            ]
            !=
            run_fingerprint
        ):

            raise RuntimeError(
                "Outer refit fingerprint mismatch."
            )


        return


    seed_everything(
        run_seed
    )


    training_loader, training_generator = (
        build_training_loader(
            training_dataframe,
            run_seed,
        )
    )


    resume_payload = None


    if last_checkpoint_path.exists():

        resume_payload = safe_torch_load(
            last_checkpoint_path,
            map_location="cpu",
        )

        validate_outer_checkpoint_fingerprint(resume_payload, run_fingerprint)


    model = timm.create_model(
        candidate[
            "model_name"
        ],
        pretrained=(
            resume_payload
            is
            None
        ),
        num_classes=(
            NUM_CLASSES
        ),
        drop_rate=(
            DROP_RATE
        ),
        drop_path_rate=(
            DROP_PATH_RATE
        ),
    ).to(
        DEVICE
    )


    class_weights = (
        calculate_image_class_weights(
            training_dataframe,
            DEVICE,
        )
    )


    criterion = CombinedNestedLoss(
        class_weights=(
            class_weights
        ),
        ordinal_kind=(
            candidate[
                "ordinal_kind"
            ]
        ),
        ordinal_alpha=(
            candidate[
                "ordinal_alpha"
            ]
        ),
        ordinal_weight=(
            candidate[
                "ordinal_weight"
            ]
        ),
    ).to(
        DEVICE
    )


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=(
            LEARNING_RATE
        ),
        weight_decay=(
            WEIGHT_DECAY
        ),
    )


    scheduler = build_scheduler(
        optimizer
    )


    scaler = create_scaler()


    start_epoch = 1

    history = []


    if resume_payload is not None:

        model.load_state_dict(
            resume_payload[
                "model_state_dict"
            ]
        )


        optimizer.load_state_dict(
            resume_payload[
                "optimizer_state_dict"
            ]
        )


        optimizer_to_device(
            optimizer,
            DEVICE,
        )


        scheduler.load_state_dict(
            resume_payload[
                "scheduler_state_dict"
            ]
        )


        scaler.load_state_dict(
            resume_payload[
                "scaler_state_dict"
            ]
        )


        start_epoch = (
            int(
                resume_payload[
                    "completed_epoch"
                ]
            )
            +
            1
        )


        history = list(
            resume_payload[
                "history"
            ]
        )


        training_generator.set_state(
            as_cpu_byte_tensor(
                resume_payload[
                    "training_generator_state"
                ]
            )
        )


        restore_rng_state(
            resume_payload[
                "rng_state"
            ]
        )


    print(
        "\nOUTER REFIT |",
        "fold=",
        outer_fold,
        "|",
        family_name,
        "| seed=",
        nominal_seed,
        "| epochs=",
        refit_epochs,
    )


    for epoch in range(
        start_epoch,
        refit_epochs
        +
        1,
    ):

        training_metrics = train_one_epoch(
            model,
            training_loader,
            criterion,
            optimizer,
            scaler,
            DEVICE,
        )


        history.append(
            {
                "epoch": (
                    epoch
                ),
                "training_loss": (
                    training_metrics[
                        "loss"
                    ]
                ),
                "training_image_qwk": (
                    training_metrics[
                        "qwk"
                    ]
                ),
            }
        )


        atomic_csv_save(
            pd.DataFrame(
                history
            ),
            history_path,
            index=False,
        )


        scheduler.step()


        atomic_torch_save(
            {
                "run_fingerprint": (
                    run_fingerprint
                ),
                "completed_epoch": (
                    epoch
                ),
                "history": (
                    history
                ),
                "model_state_dict": (
                    model.state_dict()
                ),
                "optimizer_state_dict": (
                    optimizer.state_dict()
                ),
                "scheduler_state_dict": (
                    scheduler.state_dict()
                ),
                "scaler_state_dict": (
                    scaler.state_dict()
                ),
                "rng_state": (
                    capture_rng_state()
                ),
                "training_generator_state": (
                    training_generator
                    .get_state()
                    .cpu()
                ),
            },
            last_checkpoint_path,
        )


    blind_loader, _ = (
        build_blind_outer_loader(
            blind_outer_dataframe,
            run_seed,
        )
    )


    blind_predictions = (
        predict_blind_outer_images(
            model,
            blind_loader,
            DEVICE,
        )
    )


    blind_predictions[
        "outer_fold"
    ] = outer_fold


    blind_predictions[
        "family"
    ] = family_name


    blind_predictions[
        "candidate_id"
    ] = candidate_id


    blind_predictions[
        "nominal_seed"
    ] = nominal_seed


    blind_predictions[
        "selected_aggregation_rule"
    ] = (
        selected_aggregation_rule
    )


    atomic_csv_save(
        blind_predictions,
        blind_prediction_path,
        index=False,
    )


    completed_record = {
        "completed_utc": (
            utc_now()
        ),
        "run_fingerprint": (
            run_fingerprint
        ),
        "analysis_plan_fingerprint": (
            ANALYSIS_PLAN_FINGERPRINT
        ),
        "outer_fold": (
            outer_fold
        ),
        "family": (
            family_name
        ),
        "candidate_id": (
            candidate_id
        ),
        "nominal_seed": (
            nominal_seed
        ),
        "refit_epochs": (
            refit_epochs
        ),
        "blind_prediction_sha256": (
            file_sha256(
                blind_prediction_path
            )
        ),
        "outer_prediction_contains_true_labels": False,
        "official_partition_loaded": False,
    }


    atomic_json_save(
        completed_record,
        completed_path,
    )


    if not KEEP_COMPLETED_OUTER_CHECKPOINTS:

        try:

            if last_checkpoint_path.exists():

                last_checkpoint_path.unlink()

        except Exception:

            pass


    del model

    del optimizer

    del scheduler

    del scaler

    del criterion

    del training_loader

    del blind_loader

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ==================================================================================================
# 33. Complete one outer fold
# ==================================================================================================

def run_one_outer_fold(
    outer_fold,
):

    selection = (
        select_within_outer_training_set(
            outer_fold
        )
    )


    for family_name in [
        "ViT",
        "Swin",
    ]:

        candidate_id = (
            selection[
                "selected_candidate_by_family"
            ][
                family_name
            ]
        )


        for seed in CONFIRMATION_SEEDS:

            refit_epochs = int(
                selection[
                    "refit_epochs_by_family_and_seed"
                ][
                    family_name
                ][
                    str(
                        seed
                    )
                ]
            )


            run_outer_refit_blind(
                outer_fold,
                family_name,
                candidate_id,
                seed,
                refit_epochs,
                selection[
                    "selected_aggregation_rule"
                ],
                selection[
                    "selection_record_fingerprint"
                ],
            )


    atomic_json_save(
        {
            "completed_utc": (
                utc_now()
            ),
            "analysis_plan_fingerprint": (
                ANALYSIS_PLAN_FINGERPRINT
            ),
            "outer_fold": (
                outer_fold
            ),
            "n_completed_outer_refits": (
                6
            ),
            "outer_predictions_contain_labels": False,
            "publication_metrics_generated": False,
        },
        outer_fold_completion_path(
            outer_fold
        ),
    )


    print(
        "\nOUTER FOLD",
        outer_fold,
        "COMPLETED"
    )


# ==================================================================================================
# 34. Attach frozen labels during final assembly only
# ==================================================================================================

def normalize_path(
    value,
):

    return re.sub(
        r"/+",
        "/",
        str(
            value
        )
        .replace(
            "\\",
            "/",
        )
        .strip(),
    )

def canonical_integer_column(
    dataframe,
    column,
):

    dataframe[
        column
    ] = pd.to_numeric(
        dataframe[
            column
        ],
        errors="raise",
    )


    numeric_values = dataframe[
        column
    ].to_numpy(
        dtype=float
    )


    if not np.allclose(
        numeric_values,
        np.round(
            numeric_values
        ),
    ):

        raise RuntimeError(
            f"{column} contains non-integer values."
        )


    dataframe[
        column
    ] = dataframe[
        column
    ].astype(
        int
    )

def attach_labels(
    blind_dataframe,
):

    blind_dataframe = (
        blind_dataframe.copy()
    )


    forbidden_columns = {
        "label",
        "y_true",
        "patient_y_true",
    }


    present_forbidden = (
        forbidden_columns
        &
        set(
            blind_dataframe.columns
        )
    )


    if present_forbidden:

        raise RuntimeError(
            "Input outer prediction dataframe is not label blind.\n"
            f"Unexpected columns: {sorted(present_forbidden)}"
        )


    required_columns = {
        "relative_path",
        "patient_id",
        "cv_fold",
        "y_pred",
        *PROBABILITY_COLUMNS,
    }


    missing_columns = (
        required_columns
        -
        set(
            blind_dataframe.columns
        )
    )


    if missing_columns:

        raise RuntimeError(
            "Blind prediction dataframe missing columns:\n"
            f"{sorted(missing_columns)}\n\n"
            f"Available columns:\n"
            f"{list(blind_dataframe.columns)}"
        )


    blind_dataframe[
        "relative_path"
    ] = blind_dataframe[
        "relative_path"
    ].astype(
        str
    ).map(
        normalize_path
    )


    for integer_column in [
        "patient_id",
        "cv_fold",
        "y_pred",
    ]:

        canonical_integer_column(
            blind_dataframe,
            integer_column,
        )


    for probability_column in PROBABILITY_COLUMNS:

        blind_dataframe[
            probability_column
        ] = pd.to_numeric(
            blind_dataframe[
                probability_column
            ],
            errors="raise",
        ).astype(
            float
        )


    if blind_dataframe[
        "relative_path"
    ].duplicated().any():

        raise RuntimeError(
            "Blind prediction file contains duplicate relative paths."
        )


    probability_array = blind_dataframe[
        PROBABILITY_COLUMNS
    ].to_numpy(
        dtype=float
    )


    if not np.isfinite(
        probability_array
    ).all():

        raise RuntimeError(
            "Blind prediction file contains non-finite probabilities."
        )


    vault_for_join = (
        outer_label_vault_df.rename(
            columns={
                "patient_id": (
                    "_vault_patient_id"
                ),
                "cv_fold": (
                    "_vault_cv_fold"
                ),
            }
        )
        .copy()
    )


    input_row_count = len(
        blind_dataframe
    )


    merged = blind_dataframe.merge(
        vault_for_join,
        on="relative_path",
        how="inner",
        validate="one_to_one",
    )


    if len(
        merged
    ) != input_row_count:

        unmatched_paths = sorted(
            set(
                blind_dataframe[
                    "relative_path"
                ]
            )
            -
            set(
                merged[
                    "relative_path"
                ]
            )
        )


        print(
            "\nExample unmatched paths:"
        )


        print(
            unmatched_paths[
                :20
            ]
        )


        raise RuntimeError(
            "Final label join changed the number of prediction rows.\n"
            f"Before: {input_row_count}\n"
            f"After: {len(merged)}"
        )


    patient_id_mismatch = (
        merged[
            "patient_id"
        ].astype(
            int
        )
        !=
        merged[
            "_vault_patient_id"
        ].astype(
            int
        )
    )


    if patient_id_mismatch.any():

        display(
            merged.loc[
                patient_id_mismatch,
                [
                    "relative_path",
                    "patient_id",
                    "_vault_patient_id",
                ],
            ].head(
                20
            )
        )


        raise RuntimeError(
            "Blind prediction patient_id disagrees with frozen manifest."
        )


    fold_mismatch = (
        merged[
            "cv_fold"
        ].astype(
            int
        )
        !=
        merged[
            "_vault_cv_fold"
        ].astype(
            int
        )
    )


    if fold_mismatch.any():

        display(
            merged.loc[
                fold_mismatch,
                [
                    "relative_path",
                    "patient_id",
                    "cv_fold",
                    "_vault_cv_fold",
                ],
            ].head(
                20
            )
        )


        raise RuntimeError(
            "Blind prediction cv_fold disagrees with frozen manifest."
        )


    merged[
        "y_true"
    ] = merged[
        "label"
    ].astype(
        int
    )


    merged[
        "patient_y_true"
    ] = merged[
        "patient_y_true"
    ].astype(
        int
    )


    merged = merged.drop(
        columns=[
            "_vault_patient_id",
            "_vault_cv_fold",
        ]
    )


    forbidden_suffix_columns = {
        "patient_id_x",
        "patient_id_y",
        "cv_fold_x",
        "cv_fold_y",
        "patient_y_true_x",
        "patient_y_true_y",
    }


    unexpected_suffix_columns = (
        forbidden_suffix_columns
        &
        set(
            merged.columns
        )
    )


    if unexpected_suffix_columns:

        raise RuntimeError(
            "Unexpected merge suffix columns remain:\n"
            f"{sorted(unexpected_suffix_columns)}"
        )


    if "patient_id" not in merged.columns:

        raise RuntimeError(
            "patient_id disappeared during final label attachment."
        )


    if "cv_fold" not in merged.columns:

        raise RuntimeError(
            "cv_fold disappeared during final label attachment."
        )


    if merged[
        "patient_id"
    ].isna().any():

        raise RuntimeError(
            "Missing patient IDs after label attachment."
        )


    if merged[
        "y_true"
    ].isna().any():

        raise RuntimeError(
            "Missing image labels after label attachment."
        )


    if merged[
        "patient_y_true"
    ].isna().any():

        raise RuntimeError(
            "Missing patient references after label attachment."
        )


    return merged


# ==================================================================================================
# 35. Load completed outer predictions
# ==================================================================================================

def load_outer_predictions():

    for outer_fold in range(NUM_OUTER_FOLDS):
        validate_completed_outer_fold(outer_fold)

    selections = {}

    prediction_lookup = {}


    artifact_rows = []


    for outer_fold in range(
        NUM_OUTER_FOLDS
    ):

        selection_path = (
            INNER_ROOT
            /
            f"outer_fold_{outer_fold}"
            /
            "selection_records"
            /
            "outer_fold_inner_selection_record.json"
        )


        if not selection_path.exists():

            raise FileNotFoundError(
                f"Missing selection record:\n"
                f"{selection_path}"
            )


        with open(
            selection_path,
            "r",
            encoding="utf-8",
        ) as file:

            selection = json.load(
                file
            )


        if (
            selection.get(
                "analysis_plan_fingerprint"
            )
            !=
            ANALYSIS_PLAN_FINGERPRINT
        ):

            raise RuntimeError(
                f"Outer fold {outer_fold} selection fingerprint mismatch."
            )


        if bool(
            selection.get(
                "outer_labels_used_for_any_selection",
                True,
            )
        ):

            raise RuntimeError(
                f"Outer fold {outer_fold} selection record does not "
                "confirm label-blind outer selection."
            )


        required_selection_keys = {
            "selected_aggregation_rule",
            "selected_vit_candidate_id",
            "selected_swin_candidate_id",
            "selected_system",
            "selected_candidate_by_family",
        }


        missing_selection_keys = (
            required_selection_keys
            -
            set(
                selection.keys()
            )
        )


        if missing_selection_keys:

            raise RuntimeError(
                f"Outer fold {outer_fold} selection record missing keys:\n"
                f"{sorted(missing_selection_keys)}"
            )


        if selection[
            "selected_system"
        ] not in {
            "ViT",
            "Swin",
        }:

            raise RuntimeError(
                f"Outer fold {outer_fold} selected_system is invalid."
            )


        selections[
            outer_fold
        ] = selection


        for family_name in [
            "ViT",
            "Swin",
        ]:

            candidate_id = (
                selection[
                    "selected_candidate_by_family"
                ][
                    family_name
                ]
            )


            for seed in CONFIRMATION_SEEDS:

                prediction_path = (
                    outer_refit_directory(
                        outer_fold,
                        family_name,
                        candidate_id,
                        seed,
                    )
                    /
                    "outer_blind_image_predictions.csv"
                )


                if not prediction_path.exists():

                    raise FileNotFoundError(
                        "Missing completed blind outer prediction:\n"
                        f"{prediction_path}\n\n"
                        "This cell will NOT retrain it."
                    )


                blind_dataframe = pd.read_csv(
                    prediction_path
                )


                labeled_dataframe = attach_labels(
                    blind_dataframe
                )


                expected_image_count = int(
                    (
                        development_df[
                            "cv_fold"
                        ]
                        ==
                        outer_fold
                    ).sum()
                )


                expected_patient_count = int(
                    development_df.loc[
                        development_df[
                            "cv_fold"
                        ]
                        ==
                        outer_fold,
                        "patient_id",
                    ].nunique()
                )


                if len(
                    labeled_dataframe
                ) != expected_image_count:

                    raise RuntimeError(
                        f"Outer fold {outer_fold}, {family_name}, seed {seed}: "
                        f"image count mismatch."
                    )


                if (
                    labeled_dataframe[
                        "patient_id"
                    ].nunique()
                    !=
                    expected_patient_count
                ):

                    raise RuntimeError(
                        f"Outer fold {outer_fold}, {family_name}, seed {seed}: "
                        f"patient count mismatch."
                    )


                fold_values = sorted(
                    labeled_dataframe[
                        "cv_fold"
                    ].unique()
                    .tolist()
                )


                if fold_values != [
                    outer_fold
                ]:

                    raise RuntimeError(
                        f"Outer fold {outer_fold}, {family_name}, seed {seed}: "
                        f"unexpected cv_fold values {fold_values}."
                    )


                prediction_lookup[
                    (
                        outer_fold,
                        family_name,
                        seed,
                    )
                ] = (
                    labeled_dataframe
                )


                artifact_rows.append(
                    {
                        "outer_fold": (
                            outer_fold
                        ),
                        "family": (
                            family_name
                        ),
                        "candidate_id": (
                            candidate_id
                        ),
                        "nominal_seed": (
                            seed
                        ),
                        "n_images": int(
                            len(
                                labeled_dataframe
                            )
                        ),
                        "n_patients": int(
                            labeled_dataframe[
                                "patient_id"
                            ].nunique()
                        ),
                        "prediction_path": str(
                            prediction_path
                        ),
                        "sha256": (
                            file_sha256(
                                prediction_path
                            )
                        ),
                    }
                )


    artifact_df = pd.DataFrame(
        artifact_rows
    )


    if len(
        artifact_df
    ) != 30:

        raise RuntimeError(
            f"Expected 30 saved outer prediction files; "
            f"validated {len(artifact_df)}."
        )


    atomic_csv_save(
        artifact_df,
        MANIFEST_ROOT
        /
        "finalization_validated_outer_prediction_manifest.csv",
        index=False,
    )


    print(
        "\nAll 30 saved blind outer prediction files validated."
    )


    return (
        selections,
        prediction_lookup,
    )


# ==================================================================================================
# 36. Pooled method predictions
# ==================================================================================================

def assemble_method_seed(
    method_name,
    seed,
    selections,
    prediction_lookup,
):

    image_parts = []

    patient_parts = []


    for outer_fold in range(
        NUM_OUTER_FOLDS
    ):

        selection = (
            selections[
                outer_fold
            ]
        )


        family = (
            selection[
                "selected_system"
            ]
            if method_name
            ==
            "NestedSelected"
            else
            method_name
        )


        image_dataframe = (
            prediction_lookup[
                (
                    outer_fold,
                    family,
                    seed,
                )
            ]
            .copy()
        )


        image_dataframe[
            "outer_fold"
        ] = outer_fold


        image_dataframe[
            "nested_method"
        ] = method_name


        patient_dataframe = (
            aggregate_image_predictions(
                image_dataframe,
                selection[
                    "selected_aggregation_rule"
                ],
            )
        )


        patient_dataframe[
            "outer_fold"
        ] = outer_fold


        patient_dataframe[
            "nested_method"
        ] = method_name


        patient_dataframe[
            "nominal_seed"
        ] = seed


        image_parts.append(
            image_dataframe
        )


        patient_parts.append(
            patient_dataframe
        )


    return (
        pd.concat(
            image_parts,
            ignore_index=True,
        ),
        pd.concat(
            patient_parts,
            ignore_index=True,
        ),
    )


# ==================================================================================================
# 37. Bootstrap
# ==================================================================================================

def stratified_bootstrap(
    patient_dataframe,
    repetitions,
    seed,
):

    dataframe = (
        patient_dataframe.sort_values(
            "patient_id"
        )
        .reset_index(
            drop=True
        )
    )


    y_true = dataframe[
        "y_true"
    ].to_numpy(
        dtype=int
    )


    y_pred = dataframe[
        "y_pred"
    ].to_numpy(
        dtype=int
    )


    class_indices = {
        class_id: np.flatnonzero(
            y_true
            ==
            class_id
        )
        for class_id in CLASS_IDS
    }


    rng = np.random.default_rng(
        seed
    )


    rows = []


    for bootstrap_index in range(
        repetitions
    ):

        sampled = np.concatenate(
            [
                rng.choice(
                    class_indices[
                        class_id
                    ],
                    size=len(
                        class_indices[
                            class_id
                        ]
                    ),
                    replace=True,
                )
                for class_id in CLASS_IDS
            ]
        )


        metrics = compute_metrics(
            y_true[
                sampled
            ],
            y_pred[
                sampled
            ],
        )


        rows.append(
            {
                "bootstrap_index": (
                    bootstrap_index
                ),
                **metrics,
            }
        )


    return pd.DataFrame(
        rows
    )


# ==================================================================================================
# 38. Final assembly
# ==================================================================================================

def assemble_final_nested_results():

    selections, prediction_lookup = (
        load_outer_predictions()
    )


    methods = [
        "ViT",
        "Swin",
        "NestedSelected",
    ]


    metric_rows = []


    method_seed_patient_lookup = {}


    for method_name in methods:

        method_directory = (
            FINAL_ROOT
            /
            method_name
        )


        method_directory.mkdir(
            parents=True,
            exist_ok=True,
        )


        for seed in CONFIRMATION_SEEDS:

            images, patients = (
                assemble_method_seed(
                    method_name,
                    seed,
                    selections,
                    prediction_lookup,
                )
            )


            image_metrics = compute_metrics(
                images[
                    "y_true"
                ],
                images[
                    "y_pred"
                ],
            )


            patient_metrics = compute_metrics(
                patients[
                    "y_true"
                ],
                patients[
                    "y_pred"
                ],
            )


            method_seed_patient_lookup[
                (
                    method_name,
                    seed,
                )
            ] = patients


            metric_rows.append(
                {
                    "nested_method": (
                        method_name
                    ),
                    "nominal_seed": (
                        seed
                    ),
                    "patient_qwk": (
                        patient_metrics[
                            "qwk"
                        ]
                    ),
                    "patient_mae": (
                        patient_metrics[
                            "mae"
                        ]
                    ),
                    "patient_macro_f1": (
                        patient_metrics[
                            "macro_f1"
                        ]
                    ),
                    "patient_accuracy": (
                        patient_metrics[
                            "accuracy"
                        ]
                    ),
                    "patient_balanced_accuracy": (
                        patient_metrics[
                            "balanced_accuracy"
                        ]
                    ),
                    "image_qwk": (
                        image_metrics[
                            "qwk"
                        ]
                    ),
                    "image_mae": (
                        image_metrics[
                            "mae"
                        ]
                    ),
                }
            )


            atomic_csv_save(
                images,
                method_directory
                /
                f"nested_outer_image_predictions_seed_{seed}.csv",
                index=False,
            )


            atomic_csv_save(
                patients,
                method_directory
                /
                f"nested_outer_patient_predictions_seed_{seed}.csv",
                index=False,
            )


    metrics_by_seed_df = pd.DataFrame(
        metric_rows
    )


    atomic_csv_save(
        metrics_by_seed_df,
        FINAL_ROOT
        /
        "nested_metrics_by_seed.csv",
        index=False,
    )


    multiseed_summary_df = (
        metrics_by_seed_df.groupby(
            "nested_method",
            as_index=False,
        )
        .agg(
            patient_qwk_mean=(
                "patient_qwk",
                "mean",
            ),
            patient_qwk_std=(
                "patient_qwk",
                "std",
            ),
            patient_mae_mean=(
                "patient_mae",
                "mean",
            ),
            patient_mae_std=(
                "patient_mae",
                "std",
            ),
            patient_macro_f1_mean=(
                "patient_macro_f1",
                "mean",
            ),
            patient_macro_f1_std=(
                "patient_macro_f1",
                "std",
            ),
            patient_accuracy_mean=(
                "patient_accuracy",
                "mean",
            ),
            patient_accuracy_std=(
                "patient_accuracy",
                "std",
            ),
            image_qwk_mean=(
                "image_qwk",
                "mean",
            ),
            image_qwk_std=(
                "image_qwk",
                "std",
            ),
        )
    )


    atomic_csv_save(
        multiseed_summary_df,
        FINAL_ROOT
        /
        "nested_multiseed_summary.csv",
        index=False,
    )


    # Patient bootstrap for each method using seed 42 patient predictions.
    bootstrap_summary_rows = []


    for method_index, method_name in enumerate(
        methods
    ):

        patient_dataframe = (
            method_seed_patient_lookup[
                (
                    method_name,
                    CONFIRMATION_SEEDS[
                        0
                    ],
                )
            ]
        )


        bootstrap_samples = stratified_bootstrap(
            patient_dataframe,
            BOOTSTRAP_REPLICATES,
            BOOTSTRAP_SEED
            +
            method_index
            *
            10000,
        )


        for metric_name in METRIC_NAMES:

            values = (
                bootstrap_samples[
                    metric_name
                ]
                .dropna()
                .to_numpy()
            )


            bootstrap_summary_rows.append(
                {
                    "nested_method": (
                        method_name
                    ),
                    "metric": (
                        metric_name
                    ),
                    "ci_low": float(
                        np.percentile(
                            values,
                            2.5,
                        )
                    ),
                    "ci_high": float(
                        np.percentile(
                            values,
                            97.5,
                        )
                    ),
                    "n_valid": int(
                        len(
                            values
                        )
                    ),
                }
            )


    atomic_csv_save(
        pd.DataFrame(
            bootstrap_summary_rows
        ),
        FINAL_ROOT
        /
        "nested_bootstrap_summary.csv",
        index=False,
    )


    # Selection records.
    selection_rows = []


    for outer_fold, selection in selections.items():

        selection_rows.append(
            {
                "outer_fold": (
                    outer_fold
                ),
                "selected_aggregation_rule": (
                    selection[
                        "selected_aggregation_rule"
                    ]
                ),
                "selected_vit_candidate_id": (
                    selection[
                        "selected_vit_candidate_id"
                    ]
                ),
                "selected_swin_candidate_id": (
                    selection[
                        "selected_swin_candidate_id"
                    ]
                ),
                "selected_system": (
                    selection[
                        "selected_system"
                    ]
                ),
            }
        )


    atomic_csv_save(
        pd.DataFrame(
            selection_rows
        ),
        FINAL_ROOT
        /
        "outer_fold_selection_records.csv",
        index=False,
    )


    publication_table = (
        multiseed_summary_df[
            [
                "nested_method",
                "patient_qwk_mean",
                "patient_qwk_std",
                "patient_mae_mean",
                "patient_mae_std",
                "patient_macro_f1_mean",
                "patient_macro_f1_std",
                "patient_accuracy_mean",
                "patient_accuracy_std",
            ]
        ]
        .copy()
    )


    atomic_csv_save(
        publication_table,
        FINAL_ROOT
        /
        "table_nested_patient_grouped_evaluation.csv",
        index=False,
    )


    atomic_json_save(
        {
            "completed_utc": (
                utc_now()
            ),
            "analysis_plan_fingerprint": (
                ANALYSIS_PLAN_FINGERPRINT
            ),
            "outer_predictions_were_label_blind_until_final_assembly": True,
            "official_partition_loaded": False,
            "outer_fold_used_for_selection": False,
            "historical_locked_official_analysis_changed": False,
            "status": (
                "completed"
            ),
        },
        V2_ROOT
        /
        "RUN_COMPLETED.json",
    )


    print(
        "\n"
        +
        "=" * 125
    )


    print(
        "FINAL NESTED ASSEMBLY COMPLETED"
    )


    print(
        "=" * 125
    )


    display(
        metrics_by_seed_df
    )


    display(
        multiseed_summary_df
    )


# ==================================================================================================
# 39. Workload
# ==================================================================================================

INNER_FITS_PER_OUTER = 72

OUTER_REFITS_PER_OUTER = 6

TOTAL_INNER_FITS = (
    INNER_FITS_PER_OUTER
    *
    NUM_OUTER_FOLDS
)

TOTAL_OUTER_REFITS = (
    OUTER_REFITS_PER_OUTER
    *
    NUM_OUTER_FOLDS
)

TOTAL_MODEL_FITS = (
    TOTAL_INNER_FITS
    +
    TOTAL_OUTER_REFITS
)


print(
    "\n"
    +
    "=" * 125
)


print(
    "NESTED EVALUATION V2 WORKLOAD"
)


print(
    "=" * 125
)


print(
    "Maximum inner fits:",
    TOTAL_INNER_FITS,
)


print(
    "Maximum outer refits:",
    TOTAL_OUTER_REFITS,
)


print(
    "Maximum total model fits:",
    TOTAL_MODEL_FITS,
)


# ==================================================================================================
# 40. AUTOMATIC EXECUTION DISPATCHER
# ==================================================================================================

if EXECUTION_MODE == "PLAN":

    print(
        "\nPLAN mode only."
    )


elif EXECUTION_MODE == "RUN":

    folds_to_run = sorted(
        set(
            int(
                fold_id
            )
            for fold_id in (
                OUTER_FOLDS_TO_RUN
            )
        )
    )


    print(
        "\n"
        +
        "=" * 125
    )


    print(
        "AUTOMATIC OUTER-FOLD EXECUTION"
    )


    print(
        "=" * 125
    )


    print(
        "Requested folds:",
        folds_to_run,
    )


    for outer_fold in folds_to_run:

        if outer_fold_completion_path(
            outer_fold
        ).exists():

            validate_completed_outer_fold(outer_fold)

            print(
                f"\nOuter fold {outer_fold} already completed. "
                "Skipping."
            )

            continue


        print(
            "\n"
            +
            "=" * 125
        )


        print(
            f"STARTING OUTER FOLD {outer_fold}"
        )


        print(
            "=" * 125
        )


        run_one_outer_fold(
            outer_fold
        )


        print(
            f"\nOuter fold {outer_fold} completed successfully."
        )


    if all_outer_folds_complete():

        print(
            "\n"
            +
            "=" * 125
        )


        print(
            "ALL FIVE OUTER FOLDS COMPLETED"
        )


        print(
            "=" * 125
        )


        print(
            "Beginning final label joining and assembly."
        )


        assemble_final_nested_results()


    else:

        completed_folds = [
            fold
            for fold in range(
                NUM_OUTER_FOLDS
            )
            if outer_fold_completion_path(
                fold
            ).exists()
        ]


        incomplete_folds = (
            incomplete_outer_folds()
        )


        atomic_json_save(
            {
                "created_utc": (
                    utc_now()
                ),
                "status": (
                    "partial_outer_fold_execution"
                ),
                "analysis_plan_fingerprint": (
                    ANALYSIS_PLAN_FINGERPRINT
                ),
                "completed_outer_folds": (
                    completed_folds
                ),
                "incomplete_outer_folds": (
                    incomplete_folds
                ),
                "outer_predictions_contain_labels": False,
                "publication_metrics_generated": False,
                "official_partition_loaded": False,
            },
            V2_ROOT
            /
            "PARTIAL_RUN_STATUS.json",
        )


        print(
            "\n"
            +
            "=" * 125
        )


        print(
            "RUN STOPPED BEFORE ALL FOLDS COMPLETED"
        )


        print(
            "=" * 125
        )


        print(
            "Completed folds:",
            completed_folds,
        )


        print(
            "Incomplete folds:",
            incomplete_folds,
        )


        print(
            "\nRerun THIS SAME CELL unchanged."
        )


        print(
            "Completed folds will be skipped and incomplete runs resumed."
        )


elif EXECUTION_MODE == "ASSEMBLE":

    if not all_outer_folds_complete():

        raise RuntimeError(
            "ASSEMBLE mode requires all five folds.\n"
            f"Incomplete folds: {incomplete_outer_folds()}"
        )


    assemble_final_nested_results()